# Startup Pitch Evaluation - Single Notebook Project
This notebook contains a full project snapshot merged into one document.
Each section starts with the source file path and then the file content.


## File: backend/app/__init__.py


## File: backend/app/core/config.py


In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    app_name: str = "Startup Pitch Evaluation API"
    app_version: str = "0.1.0"
    chunk_window_seconds: int = 5
    
    # Phase 0: Migration control flags
    use_heuristic_pipeline: bool = True
    use_local_transcriber: bool = False
    local_transcriber_backend: str = "faster-whisper"
    local_transcriber_model_path: str = ""
    transcriber_min_audio_quality: float = 0.35

    model_config = SettingsConfigDict(env_file=".env", env_prefix="SPE_")


settings = Settings()



## File: backend/app/main.py


In [ ]:
import logging

from fastapi import FastAPI, HTTPException

from app.core.config import settings
from app.services.inference import InferenceService
from app.schemas import BatchEvaluationRequest, BatchEvaluationResponse, EvaluationResponse, PitchInput

logger = logging.getLogger(__name__)

app = FastAPI(title=settings.app_name, version=settings.app_version)
inference_service = InferenceService(window_seconds=settings.chunk_window_seconds)

# Log startup mode
logger.info(
    f"App started | {settings.app_name} v{settings.app_version} | "
    f"use_heuristic_pipeline={settings.use_heuristic_pipeline} | "
    f"use_local_transcriber={settings.use_local_transcriber}"
)


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "service": settings.app_name, "version": settings.app_version}


@app.post("/evaluate")
def evaluate_pitch(payload: PitchInput) -> EvaluationResponse:
    return inference_service.evaluate_payload(payload)


@app.post(
    "/evaluate/batch",
    responses={400: {"description": "Bad Request: pitches list cannot be empty"}},
)
def evaluate_pitch_batch(payload: BatchEvaluationRequest) -> BatchEvaluationResponse:
    try:
        return inference_service.evaluate_batch(payload.pitches)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc



## File: backend/app/pipeline.py


In [ ]:
import logging

from app.core.config import settings
from app.schemas import (
    ChunkReport,
    DashboardSeriesPoint,
    EvaluationResponse,
    EvaluationSummary,
    InvestorDashboard,
    PitchInput,
)
from app.services.extractors import AudioFeatureExtractor, TextFeatureExtractor, VisualFeatureExtractor
from app.services.fusion import fuse_modalities
from app.services.preprocessing import temporal_synchronize_and_segment
from app.services.reporting import build_feedback, build_investor_dashboard
from app.services.risk import detect_risk_flags
from app.services.scoring import score_chunk

logger = logging.getLogger(__name__)


class StartupPitchPipeline:
    def __init__(self, window_seconds: int = 5):
        self.window_seconds = window_seconds
        self.text_extractor = TextFeatureExtractor()
        self.visual_extractor = VisualFeatureExtractor()
        self.audio_extractor = AudioFeatureExtractor()
        
        # Log active pipeline mode
        logger.info(
            "Pipeline initialized | "
            f"use_heuristic_pipeline={settings.use_heuristic_pipeline} | "
            f"use_local_transcriber={settings.use_local_transcriber}"
        )

    def evaluate(self, payload: PitchInput, request_id: str) -> EvaluationResponse:
        chunks = temporal_synchronize_and_segment(payload, self.window_seconds)
        chunk_reports: list[ChunkReport] = []

        aggregate_scores: list[float] = []
        confidence_scores: list[float] = []
        text_scores: list[float] = []
        av_scores: list[float] = []
        language_predictions: list[str] = []
        all_quantitative_scores: list[dict] = []
        modality_totals = {"text": 0.0, "visual": 0.0, "audio": 0.0}
        risk_counts: dict[str, int] = {}
        user_stage = payload.user_details.stage if payload.user_details else ""

        for chunk in chunks:
            text_features = self.text_extractor.extract(chunk.text, payload.language_hint, chunk.slide_context)
            visual_features = self.visual_extractor.extract(chunk.slide_context, chunk.chunk_id, user_stage)
            audio_features = self.audio_extractor.extract(chunk.text)
            language_predictions.append(text_features["language_detected"])

            fused = fuse_modalities(text_features, visual_features, audio_features)
            scored = score_chunk(text_features, visual_features, audio_features, fused)
            all_quantitative_scores.extend(
                {"name": item.name, "score": item.score} for item in scored["quantitative_scores"]
            )
            modality_totals["text"] += fused["attention"]["text"]
            modality_totals["visual"] += fused["attention"]["visual"]
            modality_totals["audio"] += fused["attention"]["audio"]

            text_avg = sum(m.score for m in scored["text_metrics"]) / len(scored["text_metrics"])
            av_avg = sum(m.score for m in scored["av_metrics"]) / len(scored["av_metrics"])

            text_scores.append(text_avg)
            av_scores.append(av_avg)
            aggregate_scores.append(scored["aggregate"])
            confidence_scores.append(scored["confidence"])

            risk_flags = detect_risk_flags(chunk.text, scored["aggregate"])
            for flag in risk_flags:
                risk_counts[flag] = risk_counts.get(flag, 0) + 1

            chunk_reports.append(
                ChunkReport(
                    chunk_id=chunk.chunk_id,
                    start_sec=chunk.start_sec,
                    end_sec=chunk.end_sec,
                    text_metrics=scored["text_metrics"],
                    av_metrics=scored["av_metrics"],
                    attention=fused["attention"],
                    risk_flags=risk_flags,
                    aggregate_score=scored["aggregate"],
                )
            )

        overall_score = round(sum(aggregate_scores) / max(1, len(aggregate_scores)), 2)
        confidence_score = round(sum(confidence_scores) / max(1, len(confidence_scores)), 2)

        sorted_risks = [
            risk for risk, _count in sorted(risk_counts.items(), key=lambda x: x[1], reverse=True)
        ]

        feedback = build_feedback(
            overall_score=overall_score,
            text_avg=sum(text_scores) / max(1, len(text_scores)),
            av_avg=sum(av_scores) / max(1, len(av_scores)),
            top_risks=sorted_risks,
        )

        if overall_score >= 8.0:
            investment_band = "high-potential"
        elif overall_score >= 6.0:
            investment_band = "watchlist"
        else:
            investment_band = "early-risk"

        language_detected = max(set(language_predictions), key=language_predictions.count)

        averaged_modalities = {
            key: (value / max(1, len(chunk_reports))) for key, value in modality_totals.items()
        }

        averaged_metric_scores: dict[str, list[float]] = {}
        for metric in all_quantitative_scores:
            averaged_metric_scores.setdefault(metric["name"], []).append(metric["score"])
        metric_rows = [
            {"name": name, "score": sum(values) / len(values)}
            for name, values in averaged_metric_scores.items()
        ]

        dashboard_raw = build_investor_dashboard(
            quantitative_scores=metric_rows,
            modality_attention=averaged_modalities,
            risk_counts=risk_counts,
        )

        dashboard = InvestorDashboard(
            quantitative_scores=[DashboardSeriesPoint(**item) for item in dashboard_raw["quantitative_scores"]],
            modality_weights=[DashboardSeriesPoint(**item) for item in dashboard_raw["modality_weights"]],
            risk_distribution=[DashboardSeriesPoint(**item) for item in dashboard_raw["risk_distribution"]],
        )

        summary = EvaluationSummary(
            overall_score=overall_score,
            confidence_score=confidence_score,
            investment_band=investment_band,
            language_detected=language_detected,
            strengths=feedback["strengths"],
            weaknesses=feedback["weaknesses"],
            suggestions=feedback["suggestions"],
        )

        return EvaluationResponse(
            request_id=request_id,
            summary=summary,
            chunk_reports=chunk_reports,
            dashboard=dashboard,
        )



## File: backend/app/schemas.py


In [ ]:
from pydantic import BaseModel, Field


class PitchVideoInput(BaseModel):
    file_name: str = Field(..., description="Pitch video file name")
    file_format: str = Field(default="mp4", description="Video format: mp4 or mov")
    duration_sec: int = Field(default=60, ge=5)
    transcript_text: str = Field(default="", description="Optional ASR transcript")


class SlideInput(BaseModel):
    title: str = Field(default="")
    content: str = Field(default="")


class UserDetails(BaseModel):
    founder_name: str = Field(default="")
    startup_name: str = Field(default="")
    sector: str = Field(default="")
    stage: str = Field(default="")


class PitchInput(BaseModel):
    title: str = Field(..., description="Startup name or pitch title")
    transcript: str = Field(default="", description="Full pitch transcript")
    language_hint: str = Field(default="en-ta", description="Language mix hint")
    presenter_profile: dict = Field(default_factory=dict)
    slide_text: list[str] = Field(default_factory=list)
    video: PitchVideoInput | None = Field(default=None)
    slides: list[SlideInput] = Field(default_factory=list)
    user_details: UserDetails | None = Field(default=None)


class MetricScore(BaseModel):
    name: str
    score: float
    rationale: str


class ChunkReport(BaseModel):
    chunk_id: int
    start_sec: int
    end_sec: int
    text_metrics: list[MetricScore]
    av_metrics: list[MetricScore]
    attention: dict[str, float]
    risk_flags: list[str]
    aggregate_score: float


class EvaluationSummary(BaseModel):
    overall_score: float
    confidence_score: float
    investment_band: str
    language_detected: str
    strengths: list[str]
    weaknesses: list[str]
    suggestions: list[str]


class DashboardSeriesPoint(BaseModel):
    label: str
    value: float


class InvestorDashboard(BaseModel):
    quantitative_scores: list[DashboardSeriesPoint]
    modality_weights: list[DashboardSeriesPoint]
    risk_distribution: list[DashboardSeriesPoint]


class EvaluationResponse(BaseModel):
    request_id: str
    summary: EvaluationSummary
    chunk_reports: list[ChunkReport]
    dashboard: InvestorDashboard


class BatchEvaluationRequest(BaseModel):
    pitches: list[PitchInput] = Field(default_factory=list)


class BatchEvaluationResponse(BaseModel):
    evaluations: list[EvaluationResponse]



## File: backend/app/services/audio_processor.py


In [ ]:
"""
Audio processing module for chunk extraction and feature generation.

Handles per-5-second chunk audio extraction and mel-spectrogram feature computation.
Designed for local processing with no external API calls per design constraints.
"""

import logging
from dataclasses import dataclass

logger = logging.getLogger(__name__)


@dataclass
class AudioChunkMetadata:
    """Metadata for audio chunk and computed features."""
    chunk_id: int
    start_sec: int
    end_sec: int
    duration_sec: float
    sample_rate: int
    num_samples: int
    # Mel-spectrogram features
    mel_shape: tuple[int, int]  # (n_mels, time_bins)
    mel_mean: float  # Mean energy
    mel_std: float  # Energy variance
    # Confidence and quality
    silence_ratio: float  # Proportion of silence detected
    clipping_ratio: float  # Proportion of audio clipping
    audio_quality_score: float  # 0-1 quality estimate
    audio_hash: str  # Deterministic hash for consistency
    audio_file_path: str  # Deterministic local audio chunk path


class AudioProcessor:
    """
    Extracts audio from video per 5-second chunk and computes mel features.
    
    Design:
    - Interface-first: can use librosa/scipy locally or external service
    - Current implementation: stub generating deterministic metadata
    - Mel features persisted to chunk-based directories
    """

    def __init__(self, audio_extraction_enabled: bool = False, sample_rate: int = 16000):
        """
        Initialize audio processor.
        
        Args:
            audio_extraction_enabled: If False, generates metadata only.
                If True (future), extracts audio using ffmpeg and computes mel features.
            sample_rate: Audio sample rate (Hz). Default 16kHz (Whisper standard).
        """
        self.audio_extraction_enabled = audio_extraction_enabled
        self.sample_rate = sample_rate
        self.n_mels = 64  # Mel spectrograms: 64 frequency bins
        logger.info(
            f"AudioProcessor initialized | audio_extraction_enabled={audio_extraction_enabled} | "
            f"sample_rate={sample_rate}Hz | n_mels={self.n_mels}"
        )

    def extract_audio_chunk(
        self,
        video_file_path: str,
        chunk_id: int,
        start_sec: int,
        end_sec: int,
    ) -> AudioChunkMetadata:
        """
        Extract audio from chunk window and compute mel features.
        
        Args:
            video_file_path: Path to video file
            chunk_id: Chunk identifier
            start_sec: Chunk start time (seconds)
            end_sec: Chunk end time (seconds)
        
        Returns:
            AudioChunkMetadata with extracted features and quality metrics
        """
        duration_sec = end_sec - start_sec
        num_samples = int(duration_sec * self.sample_rate)
        
        # Mel spectrogram shape: (n_mels, time_bins)
        # time_bins ≈ num_samples / hop_length; hop_length typically 512 for 16kHz
        hop_length = 512
        n_time_bins = max(1, num_samples // hop_length)
        mel_shape = (self.n_mels, n_time_bins)
        
        # Deterministic audio hash
        audio_hash = self._compute_audio_hash(video_file_path, chunk_id, start_sec, end_sec)
        audio_file_path = self._build_audio_chunk_path(video_file_path, chunk_id)
        
        if self.audio_extraction_enabled:
            # Future: actual extraction with librosa/scipy here
            logger.debug(f"Audio extraction enabled but not yet implemented for chunk {chunk_id}")
            # Stub values pending extraction
            mel_mean = 0.0
            mel_std = 0.0
            silence_ratio = 0.0
            clipping_ratio = 0.0
            quality_score = 0.5  # Unknown quality
        else:
            # Stub: generate reasonable default values
            mel_mean = 0.0
            mel_std = 0.0
            silence_ratio = 0.0  # Assume no silence for heuristic mode
            clipping_ratio = 0.0  # Assume no clipping
            quality_score = 1.0  # Assume OK in heuristic mode (will be estimated later)
            logger.debug(f"Audio extraction skipped for chunk {chunk_id} (not enabled)")
        
        return AudioChunkMetadata(
            chunk_id=chunk_id,
            start_sec=start_sec,
            end_sec=end_sec,
            duration_sec=float(duration_sec),
            sample_rate=self.sample_rate,
            num_samples=num_samples,
            mel_shape=mel_shape,
            mel_mean=mel_mean,
            mel_std=mel_std,
            silence_ratio=silence_ratio,
            clipping_ratio=clipping_ratio,
            audio_quality_score=quality_score,
            audio_hash=audio_hash,
            audio_file_path=audio_file_path,
        )

    def _compute_audio_hash(self, video_file: str, chunk_id: int, start_sec: int, end_sec: int) -> str:
        """Deterministic hash for audio metadata consistency."""
        import hashlib
        data = f"{video_file}|{chunk_id}|{start_sec}|{end_sec}|sr={self.sample_rate}"
        return hashlib.md5(data.encode()).hexdigest()[:12]

    def _build_audio_chunk_path(self, video_file: str, chunk_id: int) -> str:
        """Deterministic local path where extracted chunk audio would be stored."""
        safe_video = video_file.split("/")[-1].replace(".", "_")
        return f"audio/{safe_video}/chunk_{chunk_id:04d}.wav"



## File: backend/app/services/extractors.py


In [ ]:
from models.audio_encoder import AudioEncoder
from models.text_encoder import TextEncoder
from models.visual_encoder import VisualEncoder


class TextFeatureExtractor:
    def __init__(self) -> None:
        self.model = TextEncoder(embedding_dim=24)

    def extract(self, chunk_text: str, language_hint: str, slide_context: str) -> dict:
        return self.model.infer(chunk_text, language_hint, slide_context)


class VisualFeatureExtractor:
    def __init__(self) -> None:
        self.model = VisualEncoder(embedding_dim=24)

    def extract(self, slide_context: str, chunk_id: int, user_stage: str) -> dict:
        return self.model.infer(slide_context, chunk_id, user_stage)


class AudioFeatureExtractor:
    def __init__(self) -> None:
        self.model = AudioEncoder(embedding_dim=24)

    def extract(self, chunk_text: str) -> dict:
        return self.model.infer(chunk_text)



## File: backend/app/services/fusion.py


In [ ]:
from models.fusion_head import FusionHead


_fusion_head = FusionHead()


def fuse_modalities(text_features: dict, visual_features: dict, audio_features: dict) -> dict:
    return _fusion_head.infer(
        text_embedding=text_features["embedding"],
        visual_embedding=visual_features["embedding"],
        audio_embedding=audio_features["embedding"],
    )



## File: backend/app/services/inference.py


In [ ]:
from __future__ import annotations

import logging
import hashlib
from pathlib import Path

from app.core.config import settings
from app.pipeline import StartupPitchPipeline
from app.schemas import BatchEvaluationResponse, EvaluationResponse, PitchInput, PitchVideoInput

logger = logging.getLogger(__name__)


class InferenceService:
    """Shared inference entrypoint for CLI and FastAPI.

    This service keeps video inference behavior consistent across interfaces and
    ensures local transcriber settings flow through preprocessing.
    """

    def __init__(self, window_seconds: int | None = None) -> None:
        self.pipeline = StartupPitchPipeline(
            window_seconds=window_seconds or settings.chunk_window_seconds
        )

    @staticmethod
    def _build_request_id(payload: PitchInput) -> str:
        serialized = payload.model_dump_json()
        digest = hashlib.sha1(serialized.encode("utf-8")).hexdigest()[:16]
        return f"req-{digest}"

    def evaluate_payload(self, payload: PitchInput, request_id: str | None = None) -> EvaluationResponse:
        inference_id = request_id or self._build_request_id(payload)
        return self.pipeline.evaluate(payload, request_id=inference_id)

    def evaluate_batch(self, pitches: list[PitchInput]) -> BatchEvaluationResponse:
        if not pitches:
            raise ValueError("pitches list cannot be empty")
        evaluations = [self.evaluate_payload(pitch) for pitch in pitches]
        return BatchEvaluationResponse(evaluations=evaluations)

    def infer_video(
        self,
        video_path: str,
        duration_sec: int = 60,
        transcript_text: str = "",
        title: str = "CLI Video Inference",
        language_hint: str = "en-ta",
    ) -> EvaluationResponse:
        safe_title = title.strip() or Path(video_path).stem or "untitled-pitch"
        payload = PitchInput(
            title=safe_title,
            transcript="",
            language_hint=language_hint,
            slide_text=[],
            video=PitchVideoInput(
                file_name=video_path,
                file_format=Path(video_path).suffix.replace(".", "") or "mp4",
                duration_sec=max(5, int(duration_sec)),
                transcript_text=transcript_text,
            ),
        )

        logger.info(
            "Running shared inference | video=%s | use_local_transcriber=%s",
            video_path,
            settings.use_local_transcriber,
        )
        return self.evaluate_payload(payload)



## File: backend/app/services/preprocessing.py


In [ ]:
import logging
from dataclasses import dataclass

from app.core.config import settings
from app.schemas import PitchInput
from app.services.audio_processor import AudioChunkMetadata, AudioProcessor
from app.services.transcriber import BaseLocalTranscriber, build_local_transcriber
from app.services.video_processor import FrameExtractMetadata, VideoProcessor

logger = logging.getLogger(__name__)


@dataclass
class ChunkAlignmentMetadata:
    """Metadata tracking text/audio/visual alignment for a chunk."""
    video_metadata: FrameExtractMetadata | None
    audio_metadata: AudioChunkMetadata
    text_excerpt: str
    slide_context: str


@dataclass
class PitchChunk:
    chunk_id: int
    start_sec: int
    end_sec: int
    text: str
    audio_chunk_ref: str
    video_chunk_ref: str
    slide_context: str
    # Phase 2: Enhanced metadata for true timeline chunking
    alignment: ChunkAlignmentMetadata


def _resolve_transcript(payload: PitchInput) -> str:
    if payload.transcript.strip():
        return payload.transcript.strip()
    if payload.video and payload.video.transcript_text.strip():
        return payload.video.transcript_text.strip()
    return ""


def _resolve_slide_context(payload: PitchInput) -> str:
    if payload.slides:
        return " ".join(f"{slide.title} {slide.content}".strip() for slide in payload.slides).strip()
    return " ".join(payload.slide_text).strip()


def temporal_synchronize_and_segment(payload: PitchInput, window_seconds: int = 5) -> list[PitchChunk]:
    """
    Synchronizes transcript/video/slide context using true timeline chunking.
    
    Creates aligned chunks based on video duration, with deterministic metadata
    for text/audio/visual alignment across all modalities.
    
    Args:
        payload: PitchInput with video, transcript, and slides
        window_seconds: Chunk duration in seconds (default 5)
    
    Returns:
        List of PitchChunk with full alignment metadata
    """
    transcript = _resolve_transcript(payload)
    slide_context = _resolve_slide_context(payload)
    
    # Get video duration (true timeline source of truth)
    video_duration_sec = payload.video.duration_sec if payload.video else 60
    video_file = payload.video.file_name if payload.video else "unknown.mp4"
    logger.info(f"Temporal segmentation | video={video_file} | duration_sec={video_duration_sec}")
    
    # Initialize processors for audio/video chunk metadata
    video_processor = VideoProcessor(frame_extraction_enabled=False)
    audio_processor = AudioProcessor(audio_extraction_enabled=False)
    transcriber: BaseLocalTranscriber | None = None
    if settings.use_local_transcriber:
        transcriber = build_local_transcriber(
            backend=settings.local_transcriber_backend,
            model_path=settings.local_transcriber_model_path,
            min_audio_quality=settings.transcriber_min_audio_quality,
        )
        logger.info(
            "Local transcriber enabled | backend=%s | model_path=%s",
            settings.local_transcriber_backend,
            settings.local_transcriber_model_path or "<unset>",
        )
    
    # Distribute transcript across timeline chunks
    sentences = [s.strip() for s in transcript.replace("\n", " ").split(".") if s.strip()]
    if not sentences:
        sentences = [transcript if transcript else "No transcript available"]
    
    chunks: list[PitchChunk] = []
    num_chunks = max(1, (video_duration_sec + window_seconds - 1) // window_seconds)
    words = " ".join(sentences).split()
    words_per_chunk = max(1, len(words) // num_chunks) if words else 1
    
    for chunk_idx in range(num_chunks):
        start_sec = chunk_idx * window_seconds
        end_sec = min((chunk_idx + 1) * window_seconds, video_duration_sec)
        
        # Assign text proportionally to timeline
        text_start_word = chunk_idx * words_per_chunk
        text_end_word = min((chunk_idx + 1) * words_per_chunk, len(words))
        chunk_text = " ".join(words[text_start_word:text_end_word]) if text_start_word < len(words) else ""
        if not chunk_text:
            chunk_text = sentences[min(chunk_idx, len(sentences) - 1)].strip()
        
        # Extract video frame metadata
        video_metadata = video_processor.extract_frames_for_chunk(
            video_file_path=video_file,
            video_duration_sec=video_duration_sec,
            chunk_id=chunk_idx,
            start_sec=start_sec,
            end_sec=end_sec,
        )
        
        # Extract audio chunk metadata
        audio_metadata = audio_processor.extract_audio_chunk(
            video_file_path=video_file,
            chunk_id=chunk_idx,
            start_sec=start_sec,
            end_sec=end_sec,
        )
        
        # Build alignment metadata
        alignment = ChunkAlignmentMetadata(
            video_metadata=video_metadata,
            audio_metadata=audio_metadata,
            text_excerpt=chunk_text,
            slide_context=slide_context,
        )

        if transcriber is not None:
            transcription = transcriber.transcribe_chunk(
                audio_file_path=audio_metadata.audio_file_path,
                audio_metadata=audio_metadata,
                fallback_text=chunk_text,
                language_hint=payload.language_hint,
            )
            chunk_text = transcription.text
            alignment.text_excerpt = chunk_text
            logger.debug(
                "Chunk %s transcription | backend=%s | status=%s | confidence=%.2f",
                chunk_idx,
                transcription.backend,
                transcription.status,
                transcription.confidence,
            )
        
        chunks.append(
            PitchChunk(
                chunk_id=chunk_idx,
                start_sec=start_sec,
                end_sec=end_sec,
                text=chunk_text,
                audio_chunk_ref=f"audio_{chunk_idx}",
                video_chunk_ref=f"video_{chunk_idx}",
                slide_context=slide_context,
                alignment=alignment,
            )
        )
    
    logger.info(f"Generated {len(chunks)} aligned chunks | window_seconds={window_seconds}")
    return chunks



## File: backend/app/services/reporting.py


In [ ]:
def build_feedback(overall_score: float, text_avg: float, av_avg: float, top_risks: list[str]) -> dict:
    strengths: list[str] = []
    weaknesses: list[str] = []
    suggestions: list[str] = []

    if text_avg >= 7:
        strengths.append("Strong articulation of problem and market context")
    else:
        weaknesses.append("Narrative around problem-market fit needs sharper framing")
        suggestions.append("Open with customer pain backed by one concrete metric")

    if av_avg >= 7:
        strengths.append("Confident delivery and visual support")
    else:
        weaknesses.append("Presentation confidence and pacing can improve")
        suggestions.append("Tighten slide text and slow down key transitions")

    if overall_score >= 8:
        strengths.append("Investor readiness appears high")
    elif overall_score < 6:
        suggestions.append("Add traction proof points and a clearer business model story")

    if "competitive-landscape-missing" in top_risks:
        weaknesses.append("Competitive positioning is not clearly defended")
        suggestions.append("Add competitor matrix with clear differentiation")

    if "overclaim-risk" in top_risks:
        weaknesses.append("Claims may appear inflated to investors")
        suggestions.append("Replace absolute claims with verifiable evidence")

    return {
        "strengths": strengths,
        "weaknesses": weaknesses,
        "suggestions": suggestions,
    }


def build_investor_dashboard(
    quantitative_scores: list[dict],
    modality_attention: dict[str, float],
    risk_counts: dict[str, int],
) -> dict:
    return {
        "quantitative_scores": [
            {"label": item["name"], "value": round(item["score"], 2)}
            for item in quantitative_scores
        ],
        "modality_weights": [
            {"label": key, "value": round(value, 2)} for key, value in modality_attention.items()
        ],
        "risk_distribution": [
            {"label": key, "value": float(value)} for key, value in sorted(risk_counts.items())
        ],
    }



## File: backend/app/services/risk.py


In [ ]:
def detect_risk_flags(chunk_text: str, aggregate_score: float) -> list[str]:
    lowered = chunk_text.lower()
    flags: list[str] = []

    if aggregate_score < 5.5:
        flags.append("low-quality-signal")

    if "no competition" in lowered or "no competitor" in lowered:
        flags.append("competitive-landscape-missing")

    if "guaranteed" in lowered or "100%" in lowered:
        flags.append("overclaim-risk")

    if "soon" in lowered and "revenue" in lowered:
        flags.append("traction-evidence-weak")

    return flags



## File: backend/app/services/scoring.py


In [ ]:
from app.schemas import MetricScore
from models.scoring_head import ScoringHead


_scoring_head = ScoringHead()


_TEXT_RATIONALES = {
    "Problem Clarity": "How clearly the startup pain point is communicated.",
    "Market Opportunity": "Presence and depth of market framing.",
    "Solution Uniqueness": "Distinctiveness of the proposed solution.",
    "Traction Evidence": "Evidence of adoption, growth, pilots, or revenue.",
    "Business Model Strength": "Clarity on monetization and economics.",
    "Team Readiness": "Signals of execution capability and preparedness.",
}

_AV_RATIONALES = {
    "Delivery Clarity": "Quality of visual communication and structure.",
    "Presenter Confidence": "Confidence cues from visual stream proxies.",
    "Voice Pace": "Speech pacing suitability for investor comprehension.",
    "Voice Prosody": "Variation and emphasis in delivery.",
}


def score_chunk(text_features: dict, visual_features: dict, audio_features: dict, fused: dict) -> dict:
    outputs = _scoring_head.infer(text_features, visual_features, audio_features, fused)

    text_metrics = [
        MetricScore(name=name, score=round(score, 2), rationale=_TEXT_RATIONALES[name])
        for name, score in outputs["text_scores"].items()
    ]

    av_metrics = [
        MetricScore(name=name, score=round(score, 2), rationale=_AV_RATIONALES[name])
        for name, score in outputs["av_scores"].items()
    ]

    return {
        "text_metrics": text_metrics,
        "av_metrics": av_metrics,
        "quantitative_scores": text_metrics + av_metrics,
        "aggregate": outputs["aggregate"],
        "confidence": outputs["confidence"],
    }



## File: backend/app/services/transcriber.py


In [ ]:
"""Local transcription abstraction with deterministic fallback behavior.

This module intentionally avoids any network behavior. Backends only run when
the required package and local model artifacts are available.
"""

from __future__ import annotations

import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass
from pathlib import Path

from app.services.audio_processor import AudioChunkMetadata

logger = logging.getLogger(__name__)


@dataclass
class TranscriptionResult:
    text: str
    confidence: float
    backend: str
    status: str
    reason: str = ""


class BaseLocalTranscriber(ABC):
    def __init__(self, backend_name: str, min_audio_quality: float = 0.35):
        self.backend_name = backend_name
        self.min_audio_quality = min_audio_quality

    @abstractmethod
    def transcribe_audio_file(self, audio_file_path: str, language_hint: str = "") -> TranscriptionResult:
        """Backend-specific transcription for a local audio file."""

    def transcribe_chunk(
        self,
        audio_file_path: str,
        audio_metadata: AudioChunkMetadata,
        fallback_text: str,
        language_hint: str = "",
    ) -> TranscriptionResult:
        """Transcribe a chunk with deterministic fallback for poor/missing audio."""
        if audio_metadata.audio_quality_score < self.min_audio_quality:
            return self._fallback(
                fallback_text=fallback_text,
                confidence=0.2,
                status="fallback-low-quality",
                reason=f"audio_quality={audio_metadata.audio_quality_score:.2f}",
            )

        if audio_metadata.silence_ratio >= 0.95:
            return self._fallback(
                fallback_text=fallback_text,
                confidence=0.15,
                status="fallback-silent",
                reason=f"silence_ratio={audio_metadata.silence_ratio:.2f}",
            )

        audio_path = Path(audio_file_path)
        if not audio_path.exists() or not audio_path.is_file():
            return self._fallback(
                fallback_text=fallback_text,
                confidence=0.35,
                status="fallback-no-audio-file",
                reason=f"audio_file_missing={audio_file_path}",
            )

        result = self.transcribe_audio_file(audio_file_path, language_hint=language_hint)
        if result.text.strip():
            return result

        return self._fallback(
            fallback_text=fallback_text,
            confidence=min(result.confidence, 0.3),
            status="fallback-empty-transcript",
            reason=result.reason or "backend returned empty transcript",
        )

    def _fallback(self, fallback_text: str, confidence: float, status: str, reason: str) -> TranscriptionResult:
        text = fallback_text.strip() or "[inaudible chunk]"
        return TranscriptionResult(
            text=text,
            confidence=round(max(0.0, min(1.0, confidence)), 2),
            backend=self.backend_name,
            status=status,
            reason=reason,
        )


class WhisperLocalTranscriber(BaseLocalTranscriber):
    def __init__(
        self,
        model_path: str,
        min_audio_quality: float = 0.35,
    ):
        super().__init__(backend_name="whisper", min_audio_quality=min_audio_quality)
        self.model_path = model_path
        self._model = None

    def _load_model(self):
        if self._model is not None:
            return self._model
        if not self.model_path or not Path(self.model_path).exists():
            return None
        try:
            import whisper  # type: ignore

            self._model = whisper.load_model(self.model_path)
            return self._model
        except Exception as exc:
            logger.warning("Whisper backend unavailable: %s", exc)
            return None

    def transcribe_audio_file(self, audio_file_path: str, language_hint: str = "") -> TranscriptionResult:
        model = self._load_model()
        if model is None:
            return self._fallback(
                fallback_text="",
                confidence=0.0,
                status="backend-unavailable",
                reason="whisper model/package not available locally",
            )
        try:
            kwargs = {}
            normalized_hint = language_hint.strip().lower()
            if normalized_hint in {"en", "ta"}:
                kwargs["language"] = normalized_hint
            output = model.transcribe(audio_file_path, **kwargs)
            text = (output.get("text") or "").strip()
            confidence = 0.8 if text else 0.0
            return TranscriptionResult(
                text=text,
                confidence=confidence,
                backend=self.backend_name,
                status="ok" if text else "empty",
                reason="",
            )
        except Exception as exc:
            return TranscriptionResult(
                text="",
                confidence=0.0,
                backend=self.backend_name,
                status="error",
                reason=str(exc),
            )


class FasterWhisperLocalTranscriber(BaseLocalTranscriber):
    def __init__(
        self,
        model_path: str,
        min_audio_quality: float = 0.35,
    ):
        super().__init__(backend_name="faster-whisper", min_audio_quality=min_audio_quality)
        self.model_path = model_path
        self._model = None

    def _load_model(self):
        if self._model is not None:
            return self._model
        if not self.model_path or not Path(self.model_path).exists():
            return None
        try:
            from faster_whisper import WhisperModel  # type: ignore

            self._model = WhisperModel(self.model_path, device="cpu", compute_type="int8")
            return self._model
        except Exception as exc:
            logger.warning("faster-whisper backend unavailable: %s", exc)
            return None

    def transcribe_audio_file(self, audio_file_path: str, language_hint: str = "") -> TranscriptionResult:
        model = self._load_model()
        if model is None:
            return self._fallback(
                fallback_text="",
                confidence=0.0,
                status="backend-unavailable",
                reason="faster-whisper model/package not available locally",
            )
        try:
            normalized_hint = language_hint.strip().lower()
            language = normalized_hint if normalized_hint in {"en", "ta"} else None
            segments, info = model.transcribe(audio_file_path, language=language)
            text = " ".join(segment.text.strip() for segment in segments if segment.text).strip()
            confidence = float(getattr(info, "language_probability", 0.8)) if text else 0.0
            return TranscriptionResult(
                text=text,
                confidence=round(max(0.0, min(1.0, confidence)), 2),
                backend=self.backend_name,
                status="ok" if text else "empty",
                reason="",
            )
        except Exception as exc:
            return TranscriptionResult(
                text="",
                confidence=0.0,
                backend=self.backend_name,
                status="error",
                reason=str(exc),
            )


def build_local_transcriber(
    backend: str,
    model_path: str,
    min_audio_quality: float = 0.35,
) -> BaseLocalTranscriber:
    normalized = backend.strip().lower()
    if normalized in {"faster-whisper", "faster_whisper", "faster"}:
        return FasterWhisperLocalTranscriber(model_path=model_path, min_audio_quality=min_audio_quality)
    if normalized == "whisper":
        return WhisperLocalTranscriber(model_path=model_path, min_audio_quality=min_audio_quality)
    logger.warning("Unknown transcriber backend '%s'; defaulting to faster-whisper", backend)
    return FasterWhisperLocalTranscriber(model_path=model_path, min_audio_quality=min_audio_quality)



## File: backend/app/services/video_processor.py


In [ ]:
"""
Video processing module for frame extraction and visual feature persistence.

Handles per-5-second chunk frame extraction from video inputs.
Currently supports local video processing; external API calls disabled per design constraints.
"""

import logging
from dataclasses import dataclass
from pathlib import Path

logger = logging.getLogger(__name__)


@dataclass
class FrameExtractMetadata:
    """Metadata for extracted frames from a chunk."""
    chunk_id: int
    start_sec: int
    end_sec: int
    frame_count: int
    frame_dir: str  # Directory where frames would be persisted
    frame_hash: str  # Deterministic hash for consistency
    extraction_status: str  # "success", "pending", "skipped"


class VideoProcessor:
    """
    Extracts frames from video per 5-second chunk.
    
    Design:
    - Interface-first: real implementation uses ffmpeg or similar
    - Current implementation: stub that generates deterministic metadata
    - Frames persisted to chunk_id-based directories
    """

    def __init__(self, frame_extraction_enabled: bool = False):
        """
        Initialize video processor.
        
        Args:
            frame_extraction_enabled: If False, generates metadata only (for non-ML phases).
                If True (future), actually extracts frames using ffmpeg/cv2.
        """
        self.frame_extraction_enabled = frame_extraction_enabled
        logger.info(f"VideoProcessor initialized | frame_extraction_enabled={frame_extraction_enabled}")

    def extract_frames_for_chunk(
        self,
        video_file_path: str,
        video_duration_sec: int,
        chunk_id: int,
        start_sec: int,
        end_sec: int,
        frames_per_chunk: int = 5,
    ) -> FrameExtractMetadata:
        """
        Extract frames from chunk window.
        
        Args:
            video_file_path: Path to video file
            video_duration_sec: Total video duration
            chunk_id: Chunk identifier
            start_sec: Chunk start time (seconds)
            end_sec: Chunk end time (seconds)
            frames_per_chunk: Number of evenly-spaced frames to extract
        
        Returns:
            FrameExtractMetadata with extraction status and locations
        """
        # Deterministic frame hash based on video properties and chunk timing
        frame_hash = self._compute_frame_hash(video_file_path, chunk_id, start_sec, end_sec)
        
        frame_dir = f"frames/{video_file_path.split('/')[-1].replace('.', '_')}/chunk_{chunk_id:04d}"

        if self.frame_extraction_enabled:
            # Future: actual ffmpeg extraction here
            logger.debug(f"Frame extraction enabled but not yet implemented for chunk {chunk_id}")
            status = "pending"
        else:
            # Stub: just generate metadata for now
            status = "skipped"
            logger.debug(f"Frame extraction skipped for chunk {chunk_id} (not enabled)")

        return FrameExtractMetadata(
            chunk_id=chunk_id,
            start_sec=start_sec,
            end_sec=end_sec,
            frame_count=frames_per_chunk,
            frame_dir=frame_dir,
            frame_hash=frame_hash,
            extraction_status=status,
        )

    def _compute_frame_hash(self, video_file: str, chunk_id: int, start_sec: int, end_sec: int) -> str:
        """Deterministic hash for frame metadata consistency."""
        import hashlib
        data = f"{video_file}|{chunk_id}|{start_sec}|{end_sec}"
        return hashlib.md5(data.encode()).hexdigest()[:12]



## File: backend/models/__init__.py


In [ ]:
"""Model package for trainable multimodal components."""



## File: backend/models/audio_encoder.py


In [ ]:
import hashlib


class AudioEncoder:
    """Audio encoder proxy for pace and prosody signals."""

    def __init__(self, embedding_dim: int = 24) -> None:
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        if self.embedding_dim <= len(values):
            return values[: self.embedding_dim]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    @staticmethod
    def _clamp_10(value: float) -> float:
        return max(0.0, min(10.0, value))

    def infer(self, chunk_text: str) -> dict:
        words = chunk_text.split()
        punctuation_count = sum(1 for c in chunk_text if c in ",;:!?")
        pace = len(words) / 10.0
        prosody = 5.0 + min(punctuation_count, 8) * 0.5

        return {
            "embedding": self._hash_to_vector(f"audio::{chunk_text}"),
            "voice_pace": self._clamp_10(10.0 - abs(4.5 - pace) * 1.5),
            "prosody": self._clamp_10(prosody),
        }



## File: backend/models/checkpoints/training_cpu_checkpoint.json


In [ ]:
{
  "weights": [
    [
      -0.5551032011786199,
      -0.8662232099762509,
      0.45480666245993107,
      -1.1462199836755063,
      0.10591492444021018,
      -0.3216335664667008,
      -1.2732095138527233,
      0.10830371212741184,
      -1.2919987727111715,
      0.016809630370312106,
      -1.1529416851942456,
      -1.20442209258471,
      -0.19756678382125192,
      1.1424498335421143,
      -0.9888883552460661,
      -0.6603050146954176
    ],
    [
      0.4725707524938196,
      1.4835288719393833,
      0.13295237906199275,
      -0.33219605822196185,
      1.6360479772067202,
      -1.7054468730130494,
      1.2285827107336362,
      -0.6201136049138056,
      -1.2616356536402116,
      -1.288436451216355,
      -0.760151151615315,
      1.075818727384177,
      -1.1361061589579444,
      0.32710904811400154,
      0.463880942314625,
      -0.3662580320937248
    ],
    [
      0.23489845218144215,
      -1.5743201344447542,
      -1.5523758474098464,
      -1.0821743498321932,
      0.7781056950123553,
      -0.24166461220446403,
      -0.7357811674101195,
      0.3637690530683387,
      -0.12732779354895563,
      -0.7567992200662931,
      1.1049692924983001,
      0.7093778030379289,
      -1.0296420872582712,
      0.32620044522185115,
      0.09701686078710701,
      1.2750741488715531
    ],
    [
      0.8113714959415562,
      -0.6328927757375035,
      1.561921632186512,
      -1.3094978298866051,
      -0.34608182227453205,
      0.9002494827177121,
      -1.1321019916241568,
      -0.020306847853347593,
      -1.5465682170393056,
      0.5874140665386426,
      0.7429714913544019,
      0.3945131945250188,
      1.3157877944256358,
      -0.6807216202757107,
      0.7570199742155341,
      0.3470585192579262
    ],
    [
      0.20345934169150562,
      -0.11911762313190559,
      1.178119623166404,
      1.4873906623101696,
      -0.14096156836317558,
      0.4911212370369945,
      -1.5857285930134462,
      0.6883015905395693,
      0.46279745557147867,
      1.6069921502841502,
      0.9288970752390439,
      -0.7716608422039108,
      -0.44736178816729016,
      0.36790364805315284,
      -1.6697038260685078,
      -0.023708807800529398
    ],
    [
      -1.0954421039550406,
      -1.1893919055956175,
      -1.4371638108039735,
      1.0828841052822915,
      -1.2915050910493666,
      -0.919990885891967,
      -0.3915392350453688,
      1.3136844519089617,
      -1.4074849783798742,
      -0.022979490266382834,
      0.3015473578714594,
      1.4163914272184221,
      1.1660278964427255,
      1.186978844821276,
      -0.7277231825708241,
      -0.3895749532601233
    ],
    [
      -0.522604130384807,
      1.1944351426841244,
      1.6311276503918815,
      -1.1737268349959236,
      -1.0030685148338536,
      -0.9169502481871253,
      -1.025090717349353,
      -0.08375042043239683,
      0.24617588266413437,
      -0.8566909094627492,
      -1.7660920373593996,
      -0.24940176651196921,
      -0.31998194204516767,
      0.33541817925487244,
      1.5804581376064588,
      0.7197356348069599
    ],
    [
      0.04467210648600604,
      0.5268158287877924,
      0.46981515717486433,
      -1.5078000476456843,
      1.4917986008282649,
      0.9880463255334436,
      1.3688822156458176,
      1.1206751710156384,
      -0.2931221931010562,
      -0.3942086476285092,
      -1.4119410441196656,
      0.5377875663400864,
      -1.4727582066502896,
      -1.5164417351049293,
      -0.9424077945914034,
      -1.0868939485606544
    ],
    [
      -0.3966754988516663,
      -1.4409482459933447,
      -1.4216357338001544,
      -0.8630122103221047,
      -1.4353264873599783,
      -0.22902475415283505,
      -1.6332785933074365,
      1.000978075671591,
      -0.0517549647565423,
      -0.8291636256602783,
      -0.6665224827167696,
      -0.5253091835716712,
      -0.24262651547418335,
      -1.2051418638053208,
      0.9767926145573875,
      1.1917795436860805
    ],
    [
      0.03648532107592371,
      -0.27330143192680956,
      -1.4426175170132807,
      -1.4593707087636996,
      -0.5999266905251948,
      -0.7810956654632459,
      1.1474667861257803,
      -1.0715614337941843,
      -1.6102034923838071,
      1.6039736419894024,
      0.2197965646377756,
      -1.1987150511824762,
      0.2977562101262105,
      -1.5191867844894422,
      0.04664853822567342,
      1.530022001917439
    ]
  ],
  "biases": [
    -0.19868223818387284,
    0.6107658695400122,
    -0.7354930955658554,
    -0.33240841036232416,
    -0.5834092043398625,
    0.3046863166440123,
    0.02556837391199852,
    0.5787647352217777,
    -0.8479662773183644,
    -0.837483784013776
  ],
  "config": {
    "epochs": 8,
    "learning_rate": 0.03,
    "train_samples": 120,
    "val_samples": 48,
    "test_samples": 48,
    "seed": 7,
    "checkpoint_dir": "models/checkpoints",
    "checkpoint_name": "training_cpu_checkpoint.json",
    "device": "cpu"
  },
  "history": [
    {
      "epoch": 1,
      "train_loss": 0.9229,
      "val_mae": 0.2818,
      "val_rmse": 0.3423,
      "val_spearman": 0.2236
    },
    {
      "epoch": 2,
      "train_loss": 0.2944,
      "val_mae": 0.3172,
      "val_rmse": 0.3713,
      "val_spearman": 0.1772
    },
    {
      "epoch": 3,
      "train_loss": 0.1391,
      "val_mae": 0.337,
      "val_rmse": 0.3909,
      "val_spearman": 0.1531
    },
    {
      "epoch": 4,
      "train_loss": 0.085,
      "val_mae": 0.3471,
      "val_rmse": 0.4027,
      "val_spearman": 0.1546
    },
    {
      "epoch": 5,
      "train_loss": 0.0628,
      "val_mae": 0.3528,
      "val_rmse": 0.4098,
      "val_spearman": 0.1552
    },
    {
      "epoch": 6,
      "train_loss": 0.0524,
      "val_mae": 0.3561,
      "val_rmse": 0.4143,
      "val_spearman": 0.1562
    },
    {
      "epoch": 7,
      "train_loss": 0.047,
      "val_mae": 0.3581,
      "val_rmse": 0.4169,
      "val_spearman": 0.1566
    },
    {
      "epoch": 8,
      "train_loss": 0.0439,
      "val_mae": 0.3595,
      "val_rmse": 0.4185,
      "val_spearman": 0.1544
    }
  ]
}


## File: backend/models/config/training_cpu.yaml


In [ ]:
# Phase-5 CPU training defaults
device: cpu
epochs: 8
learning_rate: 0.03
train_samples: 120
val_samples: 48
test_samples: 48
seed: 7
checkpoint_dir: models/checkpoints
checkpoint_name: training_cpu_checkpoint.json



## File: backend/models/config/training_gpu.yaml


In [ ]:
# Phase-5 GPU profile (same trainer now, device flag reserved for future)
device: gpu
epochs: 12
learning_rate: 0.025
train_samples: 200
val_samples: 64
test_samples: 64
seed: 11
checkpoint_dir: models/checkpoints
checkpoint_name: training_gpu_checkpoint.json



## File: backend/models/fusion_head.py


In [ ]:
class FusionHead:
    """Cross-modal fusion head with adaptive attention from modality energy."""

    @staticmethod
    def _mean(values: list[float]) -> float:
        return sum(values) / max(1, len(values))

    def infer(self, text_embedding: list[float], visual_embedding: list[float], audio_embedding: list[float]) -> dict:
        text_energy = self._mean(text_embedding)
        visual_energy = self._mean(visual_embedding)
        audio_energy = self._mean(audio_embedding)

        total = max(1e-6, text_energy + visual_energy + audio_energy)
        text_w = text_energy / total
        visual_w = visual_energy / total
        audio_w = audio_energy / total

        fused_vector = [
            (text_w * t) + (visual_w * v) + (audio_w * a)
            for t, v, a in zip(text_embedding, visual_embedding, audio_embedding)
        ]

        return {
            "vector": fused_vector,
            "attention": {
                "text": round(text_w, 4),
                "visual": round(visual_w, 4),
                "audio": round(audio_w, 4),
            },
        }



## File: backend/models/scoring_head.py


In [ ]:
class ScoringHead:
    """Multi-output scoring head that maps modality outputs into final scores."""

    @staticmethod
    def _avg(values: list[float]) -> float:
        return sum(values) / max(1, len(values))

    @staticmethod
    def _clamp_10(value: float) -> float:
        return max(0.0, min(10.0, value))

    def infer(self, text_features: dict, visual_features: dict, audio_features: dict, fused: dict) -> dict:
        text_scores = {
            "Problem Clarity": float(text_features["problem_clarity"]),
            "Market Opportunity": float(text_features["market_opportunity"]),
            "Solution Uniqueness": float(text_features["solution_uniqueness"]),
            "Traction Evidence": float(text_features["traction_evidence"]),
            "Business Model Strength": float(text_features["business_model_strength"]),
            "Team Readiness": float(text_features["team_readiness"]),
        }
        av_scores = {
            "Delivery Clarity": float(visual_features["delivery_clarity"]),
            "Presenter Confidence": float(visual_features["presenter_confidence"]),
            "Voice Pace": float(audio_features["voice_pace"]),
            "Voice Prosody": float(audio_features["prosody"]),
        }

        text_avg = self._avg(list(text_scores.values()))
        av_avg = self._avg(list(av_scores.values()))
        fusion_signal = self._avg(fused["vector"]) * 10.0

        aggregate = self._clamp_10((text_avg * 0.5) + (av_avg * 0.35) + (fusion_signal * 0.15))
        confidence = self._clamp_10((av_avg * 0.5) + (text_avg * 0.5))

        return {
            "text_scores": text_scores,
            "av_scores": av_scores,
            "aggregate": round(aggregate, 2),
            "confidence": round(confidence, 2),
        }



## File: backend/models/text_encoder.py


In [ ]:
import hashlib
import re


class TextEncoder:
    """Deterministic text encoder that mimics trainable inference behavior."""

    def __init__(self, embedding_dim: int = 24) -> None:
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        if self.embedding_dim <= len(values):
            return values[: self.embedding_dim]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    def _detect_language(self, chunk_text: str, language_hint: str) -> str:
        tamil_chars = len(re.findall(r"[\u0B80-\u0BFF]", chunk_text))
        english_chars = len(re.findall(r"[A-Za-z]", chunk_text))
        hint = language_hint.lower()

        if tamil_chars > 0 and english_chars > 0:
            return "ta-en"
        if tamil_chars > 0:
            return "ta"
        if "ta" in hint and "en" in hint:
            return "ta-en"
        return "en"

    def _normalize(self, chunk_text: str) -> str:
        return " ".join(chunk_text.replace("\n", " ").split()).strip()

    @staticmethod
    def _clamp_10(value: float) -> float:
        return max(0.0, min(10.0, value))

    def infer(self, chunk_text: str, language_hint: str, slide_context: str) -> dict:
        normalized_text = self._normalize(chunk_text)
        words = normalized_text.split()
        unique_ratio = len(set(words)) / max(1, len(words))
        language = self._detect_language(normalized_text, language_hint)
        lowered = normalized_text.lower()

        problem_signal = 2.0 if "problem" in lowered else 0.0
        market_signal = 2.0 if "market" in lowered else 0.0
        traction_signal = 2.0 if any(k in lowered for k in ["pilot", "growth", "revenue"]) else 0.0
        business_model_signal = 2.0 if any(k in lowered for k in ["price", "subscription", "saas", "margin"]) else 0.0
        team_signal = 2.0 if any(k in lowered for k in ["team", "founder", "experience"]) else 0.0
        language_bonus = 1.0 if language == "ta-en" else 0.4

        return {
            "embedding": self._hash_to_vector(f"text::{normalized_text}::{slide_context}"),
            "language_detected": language,
            "problem_clarity": self._clamp_10(4.0 + problem_signal + unique_ratio * 1.5),
            "market_opportunity": self._clamp_10(4.0 + market_signal + unique_ratio * 1.3),
            "solution_uniqueness": self._clamp_10(4.5 + unique_ratio * 3.0),
            "traction_evidence": self._clamp_10(3.5 + traction_signal + min(len(words), 40) * 0.08),
            "business_model_strength": self._clamp_10(3.5 + business_model_signal + min(len(words), 30) * 0.07),
            "team_readiness": self._clamp_10(3.0 + team_signal + language_bonus),
        }



## File: backend/models/visual_encoder.py


In [ ]:
import hashlib


class VisualEncoder:
    """Visual encoder proxy for chunk-level delivery signals."""

    def __init__(self, embedding_dim: int = 24) -> None:
        self.embedding_dim = embedding_dim

    def _hash_to_vector(self, value: str) -> list[float]:
        digest = hashlib.sha256(value.encode("utf-8")).digest()
        values = [b / 255.0 for b in digest]
        if self.embedding_dim <= len(values):
            return values[: self.embedding_dim]
        repeats = (self.embedding_dim // len(values)) + 1
        return (values * repeats)[: self.embedding_dim]

    @staticmethod
    def _clamp_10(value: float) -> float:
        return max(0.0, min(10.0, value))

    def infer(self, slide_context: str, chunk_id: int, user_stage: str) -> dict:
        text = slide_context if slide_context else "no-slides"
        density = min(len(text.split()), 120) / 120.0
        stage_bonus = 0.8 if user_stage.lower() in {"seed", "series-a", "series a"} else 0.3

        return {
            "embedding": self._hash_to_vector(f"visual::{chunk_id}::{text}::{user_stage}"),
            "delivery_clarity": self._clamp_10(4.2 + density * 3.8),
            "presenter_confidence": self._clamp_10(4.0 + (chunk_id % 5) * 0.9 + stage_bonus),
        }



## File: backend/outputs/batch_json/pitch_a.json


In [ ]:
{
  "request_id": "725c87b4-000e-40c8-b4ea-2a6b389093f0",
  "summary": {
    "overall_score": 4.86,
    "confidence_score": 4.81,
    "investment_band": "early-risk",
    "language_detected": "ta-en",
    "strengths": [],
    "weaknesses": [
      "Narrative around problem-market fit needs sharper framing",
      "Presentation confidence and pacing can improve"
    ],
    "suggestions": [
      "Open with customer pain backed by one concrete metric",
      "Tighten slide text and slow down key transitions",
      "Add traction proof points and a clearer business model story"
    ]
  },
  "chunk_reports": [
    {
      "chunk_id": 0,
      "start_sec": 0,
      "end_sec": 5,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3116,
        "visual": 0.3216,
        "audio": 0.3668
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.64
    },
    {
      "chunk_id": 1,
      "start_sec": 5,
      "end_sec": 10,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3739,
        "visual": 0.3181,
        "audio": 0.308
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.75
    },
    {
      "chunk_id": 2,
      "start_sec": 10,
      "end_sec": 15,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3427,
        "visual": 0.29,
        "audio": 0.3673
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.79
    },
    {
      "chunk_id": 3,
      "start_sec": 15,
      "end_sec": 20,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3112,
        "visual": 0.3647,
        "audio": 0.3241
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.04
    },
    {
      "chunk_id": 4,
      "start_sec": 20,
      "end_sec": 25,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.9,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3315,
        "visual": 0.3233,
        "audio": 0.3452
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.06
    },
    {
      "chunk_id": 5,
      "start_sec": 25,
      "end_sec": 30,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3321,
        "visual": 0.322,
        "audio": 0.3458
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.74
    },
    {
      "chunk_id": 6,
      "start_sec": 30,
      "end_sec": 35,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3425,
        "visual": 0.3008,
        "audio": 0.3567
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.8
    },
    {
      "chunk_id": 7,
      "start_sec": 35,
      "end_sec": 40,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3156,
        "visual": 0.3557,
        "audio": 0.3287
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.94
    },
    {
      "chunk_id": 8,
      "start_sec": 40,
      "end_sec": 45,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.327,
        "visual": 0.3326,
        "audio": 0.3404
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.99
    }
  ],
  "dashboard": {
    "quantitative_scores": [
      {
        "label": "Problem Clarity",
        "value": 5.5
      },
      {
        "label": "Market Opportunity",
        "value": 5.3
      },
      {
        "label": "Solution Uniqueness",
        "value": 7.5
      },
      {
        "label": "Traction Evidence",
        "value": 3.69
      },
      {
        "label": "Business Model Strength",
        "value": 3.66
      },
      {
        "label": "Team Readiness",
        "value": 4.0
      },
      {
        "label": "Delivery Clarity",
        "value": 4.23
      },
      {
        "label": "Presenter Confidence",
        "value": 5.9
      },
      {
        "label": "Voice Pace",
        "value": 3.6
      },
      {
        "label": "Voice Prosody",
        "value": 5.0
      }
    ],
    "modality_weights": [
      {
        "label": "text",
        "value": 0.33
      },
      {
        "label": "visual",
        "value": 0.33
      },
      {
        "label": "audio",
        "value": 0.34
      }
    ],
    "risk_distribution": [
      {
        "label": "low-quality-signal",
        "value": 9.0
      }
    ]
  }
}


## File: backend/outputs/batch_json/pitch_b.json


In [ ]:
{
  "request_id": "d66c50cf-712a-4307-bb56-875bb79f81ad",
  "summary": {
    "overall_score": 4.86,
    "confidence_score": 4.81,
    "investment_band": "early-risk",
    "language_detected": "ta-en",
    "strengths": [],
    "weaknesses": [
      "Narrative around problem-market fit needs sharper framing",
      "Presentation confidence and pacing can improve"
    ],
    "suggestions": [
      "Open with customer pain backed by one concrete metric",
      "Tighten slide text and slow down key transitions",
      "Add traction proof points and a clearer business model story"
    ]
  },
  "chunk_reports": [
    {
      "chunk_id": 0,
      "start_sec": 0,
      "end_sec": 5,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3116,
        "visual": 0.3216,
        "audio": 0.3668
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.64
    },
    {
      "chunk_id": 1,
      "start_sec": 5,
      "end_sec": 10,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3739,
        "visual": 0.3181,
        "audio": 0.308
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.75
    },
    {
      "chunk_id": 2,
      "start_sec": 10,
      "end_sec": 15,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3427,
        "visual": 0.29,
        "audio": 0.3673
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.79
    },
    {
      "chunk_id": 3,
      "start_sec": 15,
      "end_sec": 20,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3112,
        "visual": 0.3647,
        "audio": 0.3241
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.04
    },
    {
      "chunk_id": 4,
      "start_sec": 20,
      "end_sec": 25,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.9,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3315,
        "visual": 0.3233,
        "audio": 0.3452
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.06
    },
    {
      "chunk_id": 5,
      "start_sec": 25,
      "end_sec": 30,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3321,
        "visual": 0.322,
        "audio": 0.3458
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.74
    },
    {
      "chunk_id": 6,
      "start_sec": 30,
      "end_sec": 35,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3425,
        "visual": 0.3008,
        "audio": 0.3567
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.8
    },
    {
      "chunk_id": 7,
      "start_sec": 35,
      "end_sec": 40,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3156,
        "visual": 0.3557,
        "audio": 0.3287
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.94
    },
    {
      "chunk_id": 8,
      "start_sec": 40,
      "end_sec": 45,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.327,
        "visual": 0.3326,
        "audio": 0.3404
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.99
    }
  ],
  "dashboard": {
    "quantitative_scores": [
      {
        "label": "Problem Clarity",
        "value": 5.5
      },
      {
        "label": "Market Opportunity",
        "value": 5.3
      },
      {
        "label": "Solution Uniqueness",
        "value": 7.5
      },
      {
        "label": "Traction Evidence",
        "value": 3.69
      },
      {
        "label": "Business Model Strength",
        "value": 3.66
      },
      {
        "label": "Team Readiness",
        "value": 4.0
      },
      {
        "label": "Delivery Clarity",
        "value": 4.23
      },
      {
        "label": "Presenter Confidence",
        "value": 5.9
      },
      {
        "label": "Voice Pace",
        "value": 3.6
      },
      {
        "label": "Voice Prosody",
        "value": 5.0
      }
    ],
    "modality_weights": [
      {
        "label": "text",
        "value": 0.33
      },
      {
        "label": "visual",
        "value": 0.33
      },
      {
        "label": "audio",
        "value": 0.34
      }
    ],
    "risk_distribution": [
      {
        "label": "low-quality-signal",
        "value": 9.0
      }
    ]
  }
}


## File: backend/outputs/batch_summary.json


In [ ]:
{
  "mode": "batch",
  "count": 2,
  "results": [
    {
      "request_id": "725c87b4-000e-40c8-b4ea-2a6b389093f0",
      "summary": {
        "overall_score": 4.86,
        "confidence_score": 4.81,
        "investment_band": "early-risk",
        "language_detected": "ta-en",
        "strengths": [],
        "weaknesses": [
          "Narrative around problem-market fit needs sharper framing",
          "Presentation confidence and pacing can improve"
        ],
        "suggestions": [
          "Open with customer pain backed by one concrete metric",
          "Tighten slide text and slow down key transitions",
          "Add traction proof points and a clearer business model story"
        ]
      },
      "chunk_reports": [
        {
          "chunk_id": 0,
          "start_sec": 0,
          "end_sec": 5,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 4.3,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3116,
            "visual": 0.3216,
            "audio": 0.3668
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.64
        },
        {
          "chunk_id": 1,
          "start_sec": 5,
          "end_sec": 10,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 5.2,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3739,
            "visual": 0.3181,
            "audio": 0.308
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.75
        },
        {
          "chunk_id": 2,
          "start_sec": 10,
          "end_sec": 15,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 6.1,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3427,
            "visual": 0.29,
            "audio": 0.3673
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.79
        },
        {
          "chunk_id": 3,
          "start_sec": 15,
          "end_sec": 20,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.0,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3112,
            "visual": 0.3647,
            "audio": 0.3241
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 5.04
        },
        {
          "chunk_id": 4,
          "start_sec": 20,
          "end_sec": 25,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.9,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3315,
            "visual": 0.3233,
            "audio": 0.3452
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 5.06
        },
        {
          "chunk_id": 5,
          "start_sec": 25,
          "end_sec": 30,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 4.3,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3321,
            "visual": 0.322,
            "audio": 0.3458
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.74
        },
        {
          "chunk_id": 6,
          "start_sec": 30,
          "end_sec": 35,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 5.2,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3425,
            "visual": 0.3008,
            "audio": 0.3567
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.8
        },
        {
          "chunk_id": 7,
          "start_sec": 35,
          "end_sec": 40,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 6.1,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3156,
            "visual": 0.3557,
            "audio": 0.3287
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.94
        },
        {
          "chunk_id": 8,
          "start_sec": 40,
          "end_sec": 45,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.0,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.327,
            "visual": 0.3326,
            "audio": 0.3404
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.99
        }
      ],
      "dashboard": {
        "quantitative_scores": [
          {
            "label": "Problem Clarity",
            "value": 5.5
          },
          {
            "label": "Market Opportunity",
            "value": 5.3
          },
          {
            "label": "Solution Uniqueness",
            "value": 7.5
          },
          {
            "label": "Traction Evidence",
            "value": 3.69
          },
          {
            "label": "Business Model Strength",
            "value": 3.66
          },
          {
            "label": "Team Readiness",
            "value": 4.0
          },
          {
            "label": "Delivery Clarity",
            "value": 4.23
          },
          {
            "label": "Presenter Confidence",
            "value": 5.9
          },
          {
            "label": "Voice Pace",
            "value": 3.6
          },
          {
            "label": "Voice Prosody",
            "value": 5.0
          }
        ],
        "modality_weights": [
          {
            "label": "text",
            "value": 0.33
          },
          {
            "label": "visual",
            "value": 0.33
          },
          {
            "label": "audio",
            "value": 0.34
          }
        ],
        "risk_distribution": [
          {
            "label": "low-quality-signal",
            "value": 9.0
          }
        ]
      }
    },
    {
      "request_id": "d66c50cf-712a-4307-bb56-875bb79f81ad",
      "summary": {
        "overall_score": 4.86,
        "confidence_score": 4.81,
        "investment_band": "early-risk",
        "language_detected": "ta-en",
        "strengths": [],
        "weaknesses": [
          "Narrative around problem-market fit needs sharper framing",
          "Presentation confidence and pacing can improve"
        ],
        "suggestions": [
          "Open with customer pain backed by one concrete metric",
          "Tighten slide text and slow down key transitions",
          "Add traction proof points and a clearer business model story"
        ]
      },
      "chunk_reports": [
        {
          "chunk_id": 0,
          "start_sec": 0,
          "end_sec": 5,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 4.3,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3116,
            "visual": 0.3216,
            "audio": 0.3668
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.64
        },
        {
          "chunk_id": 1,
          "start_sec": 5,
          "end_sec": 10,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 5.2,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3739,
            "visual": 0.3181,
            "audio": 0.308
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.75
        },
        {
          "chunk_id": 2,
          "start_sec": 10,
          "end_sec": 15,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.58,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.57,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 6.1,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.4,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3427,
            "visual": 0.29,
            "audio": 0.3673
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.79
        },
        {
          "chunk_id": 3,
          "start_sec": 15,
          "end_sec": 20,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.0,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3112,
            "visual": 0.3647,
            "audio": 0.3241
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 5.04
        },
        {
          "chunk_id": 4,
          "start_sec": 20,
          "end_sec": 25,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.9,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3315,
            "visual": 0.3233,
            "audio": 0.3452
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 5.06
        },
        {
          "chunk_id": 5,
          "start_sec": 25,
          "end_sec": 30,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 4.3,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3321,
            "visual": 0.322,
            "audio": 0.3458
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.74
        },
        {
          "chunk_id": 6,
          "start_sec": 30,
          "end_sec": 35,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 5.2,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3425,
            "visual": 0.3008,
            "audio": 0.3567
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.8
        },
        {
          "chunk_id": 7,
          "start_sec": 35,
          "end_sec": 40,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 6.1,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.3156,
            "visual": 0.3557,
            "audio": 0.3287
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.94
        },
        {
          "chunk_id": 8,
          "start_sec": 40,
          "end_sec": 45,
          "text_metrics": [
            {
              "name": "Problem Clarity",
              "score": 5.5,
              "rationale": "How clearly the startup pain point is communicated."
            },
            {
              "name": "Market Opportunity",
              "score": 5.3,
              "rationale": "Presence and depth of market framing."
            },
            {
              "name": "Solution Uniqueness",
              "score": 7.5,
              "rationale": "Distinctiveness of the proposed solution."
            },
            {
              "name": "Traction Evidence",
              "score": 3.74,
              "rationale": "Evidence of adoption, growth, pilots, or revenue."
            },
            {
              "name": "Business Model Strength",
              "score": 3.71,
              "rationale": "Clarity on monetization and economics."
            },
            {
              "name": "Team Readiness",
              "score": 4.0,
              "rationale": "Signals of execution capability and preparedness."
            }
          ],
          "av_metrics": [
            {
              "name": "Delivery Clarity",
              "score": 4.23,
              "rationale": "Quality of visual communication and structure."
            },
            {
              "name": "Presenter Confidence",
              "score": 7.0,
              "rationale": "Confidence cues from visual stream proxies."
            },
            {
              "name": "Voice Pace",
              "score": 3.7,
              "rationale": "Speech pacing suitability for investor comprehension."
            },
            {
              "name": "Voice Prosody",
              "score": 5.0,
              "rationale": "Variation and emphasis in delivery."
            }
          ],
          "attention": {
            "text": 0.327,
            "visual": 0.3326,
            "audio": 0.3404
          },
          "risk_flags": [
            "low-quality-signal"
          ],
          "aggregate_score": 4.99
        }
      ],
      "dashboard": {
        "quantitative_scores": [
          {
            "label": "Problem Clarity",
            "value": 5.5
          },
          {
            "label": "Market Opportunity",
            "value": 5.3
          },
          {
            "label": "Solution Uniqueness",
            "value": 7.5
          },
          {
            "label": "Traction Evidence",
            "value": 3.69
          },
          {
            "label": "Business Model Strength",
            "value": 3.66
          },
          {
            "label": "Team Readiness",
            "value": 4.0
          },
          {
            "label": "Delivery Clarity",
            "value": 4.23
          },
          {
            "label": "Presenter Confidence",
            "value": 5.9
          },
          {
            "label": "Voice Pace",
            "value": 3.6
          },
          {
            "label": "Voice Prosody",
            "value": 5.0
          }
        ],
        "modality_weights": [
          {
            "label": "text",
            "value": 0.33
          },
          {
            "label": "visual",
            "value": 0.33
          },
          {
            "label": "audio",
            "value": 0.34
          }
        ],
        "risk_distribution": [
          {
            "label": "low-quality-signal",
            "value": 9.0
          }
        ]
      }
    }
  ]
}


## File: backend/outputs/benchmark_runtime.json


In [ ]:
{
  "benchmark": "phase7_inference_runtime",
  "profiles": [
    {
      "profile": "cpu",
      "runs": 3,
      "avg_ms": 0.861,
      "p95_ms": 0.757,
      "min_ms": 0.697,
      "max_ms": 1.129,
      "samples_ms": [
        1.129,
        0.757,
        0.697
      ],
      "notes": "CPU baseline for current local inference stack."
    },
    {
      "profile": "gpu",
      "runs": 3,
      "avg_ms": 0.716,
      "p95_ms": 0.733,
      "min_ms": 0.649,
      "max_ms": 0.765,
      "samples_ms": [
        0.649,
        0.765,
        0.733
      ],
      "notes": "GPU profile currently uses the same deterministic local pipeline path; replace with actual GPU-backed model execution in later optimization phase."
    }
  ]
}


## File: backend/outputs/single.json


In [ ]:
{
  "request_id": "deabef3c-995c-4ed0-af0c-4b30f69309ad",
  "summary": {
    "overall_score": 4.89,
    "confidence_score": 4.85,
    "investment_band": "early-risk",
    "language_detected": "ta-en",
    "strengths": [],
    "weaknesses": [
      "Narrative around problem-market fit needs sharper framing",
      "Presentation confidence and pacing can improve"
    ],
    "suggestions": [
      "Open with customer pain backed by one concrete metric",
      "Tighten slide text and slow down key transitions",
      "Add traction proof points and a clearer business model story"
    ]
  },
  "chunk_reports": [
    {
      "chunk_id": 0,
      "start_sec": 0,
      "end_sec": 5,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3116,
        "visual": 0.3216,
        "audio": 0.3668
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.64
    },
    {
      "chunk_id": 1,
      "start_sec": 5,
      "end_sec": 10,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3739,
        "visual": 0.3181,
        "audio": 0.308
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.75
    },
    {
      "chunk_id": 2,
      "start_sec": 10,
      "end_sec": 15,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.58,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.57,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.4,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3427,
        "visual": 0.29,
        "audio": 0.3673
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.79
    },
    {
      "chunk_id": 3,
      "start_sec": 15,
      "end_sec": 20,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3112,
        "visual": 0.3647,
        "audio": 0.3241
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.04
    },
    {
      "chunk_id": 4,
      "start_sec": 20,
      "end_sec": 25,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.9,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3315,
        "visual": 0.3233,
        "audio": 0.3452
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.06
    },
    {
      "chunk_id": 5,
      "start_sec": 25,
      "end_sec": 30,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3321,
        "visual": 0.322,
        "audio": 0.3458
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.74
    },
    {
      "chunk_id": 6,
      "start_sec": 30,
      "end_sec": 35,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3425,
        "visual": 0.3008,
        "audio": 0.3567
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.8
    },
    {
      "chunk_id": 7,
      "start_sec": 35,
      "end_sec": 40,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3156,
        "visual": 0.3557,
        "audio": 0.3287
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.94
    },
    {
      "chunk_id": 8,
      "start_sec": 40,
      "end_sec": 45,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.327,
        "visual": 0.3326,
        "audio": 0.3404
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.99
    },
    {
      "chunk_id": 9,
      "start_sec": 45,
      "end_sec": 50,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.9,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3474,
        "visual": 0.2909,
        "audio": 0.3617
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.03
    },
    {
      "chunk_id": 10,
      "start_sec": 50,
      "end_sec": 55,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 4.3,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3116,
        "visual": 0.3639,
        "audio": 0.3245
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.8
    },
    {
      "chunk_id": 11,
      "start_sec": 55,
      "end_sec": 60,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 5.2,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.332,
        "visual": 0.3222,
        "audio": 0.3457
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.82
    },
    {
      "chunk_id": 12,
      "start_sec": 60,
      "end_sec": 65,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 6.1,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3304,
        "visual": 0.3255,
        "audio": 0.3441
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.91
    },
    {
      "chunk_id": 13,
      "start_sec": 65,
      "end_sec": 70,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.0,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.3467,
        "visual": 0.2923,
        "audio": 0.361
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 4.95
    },
    {
      "chunk_id": 14,
      "start_sec": 70,
      "end_sec": 75,
      "text_metrics": [
        {
          "name": "Problem Clarity",
          "score": 5.5,
          "rationale": "How clearly the startup pain point is communicated."
        },
        {
          "name": "Market Opportunity",
          "score": 5.3,
          "rationale": "Presence and depth of market framing."
        },
        {
          "name": "Solution Uniqueness",
          "score": 7.5,
          "rationale": "Distinctiveness of the proposed solution."
        },
        {
          "name": "Traction Evidence",
          "score": 3.74,
          "rationale": "Evidence of adoption, growth, pilots, or revenue."
        },
        {
          "name": "Business Model Strength",
          "score": 3.71,
          "rationale": "Clarity on monetization and economics."
        },
        {
          "name": "Team Readiness",
          "score": 4.0,
          "rationale": "Signals of execution capability and preparedness."
        }
      ],
      "av_metrics": [
        {
          "name": "Delivery Clarity",
          "score": 4.23,
          "rationale": "Quality of visual communication and structure."
        },
        {
          "name": "Presenter Confidence",
          "score": 7.9,
          "rationale": "Confidence cues from visual stream proxies."
        },
        {
          "name": "Voice Pace",
          "score": 3.7,
          "rationale": "Speech pacing suitability for investor comprehension."
        },
        {
          "name": "Voice Prosody",
          "score": 5.0,
          "rationale": "Variation and emphasis in delivery."
        }
      ],
      "attention": {
        "text": 0.324,
        "visual": 0.3386,
        "audio": 0.3374
      },
      "risk_flags": [
        "low-quality-signal"
      ],
      "aggregate_score": 5.08
    }
  ],
  "dashboard": {
    "quantitative_scores": [
      {
        "label": "Problem Clarity",
        "value": 5.5
      },
      {
        "label": "Market Opportunity",
        "value": 5.3
      },
      {
        "label": "Solution Uniqueness",
        "value": 7.5
      },
      {
        "label": "Traction Evidence",
        "value": 3.71
      },
      {
        "label": "Business Model Strength",
        "value": 3.68
      },
      {
        "label": "Team Readiness",
        "value": 4.0
      },
      {
        "label": "Delivery Clarity",
        "value": 4.23
      },
      {
        "label": "Presenter Confidence",
        "value": 6.1
      },
      {
        "label": "Voice Pace",
        "value": 3.64
      },
      {
        "label": "Voice Prosody",
        "value": 5.0
      }
    ],
    "modality_weights": [
      {
        "label": "text",
        "value": 0.33
      },
      {
        "label": "visual",
        "value": 0.32
      },
      {
        "label": "audio",
        "value": 0.34
      }
    ],
    "risk_distribution": [
      {
        "label": "low-quality-signal",
        "value": 15.0
      }
    ]
  }
}


## File: backend/requirements.txt


fastapi==0.116.1
uvicorn==0.35.0
pydantic==2.11.7
pydantic-settings==2.10.1
pytest==8.4.1
httpx==0.28.1
jupyterlab==4.3.5
ipykernel==6.29.5



## File: backend/scripts/benchmark_runtime.py


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import statistics
import sys
import time


sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from app.services.inference import InferenceService


def _run_profile(profile: str, runs: int, duration_sec: int) -> dict:
    service = InferenceService()
    timings_ms: list[float] = []

    for idx in range(runs):
        start = time.perf_counter()
        service.infer_video(
            video_path=f"benchmark_{profile}_{idx}.mp4",
            duration_sec=duration_sec,
            transcript_text="We benchmark local inference latency with deterministic synthetic input.",
            title=f"benchmark-{profile}-{idx}",
            language_hint="en-ta",
        )
        elapsed = (time.perf_counter() - start) * 1000.0
        timings_ms.append(round(elapsed, 3))

    return {
        "profile": profile,
        "runs": runs,
        "avg_ms": round(statistics.mean(timings_ms), 3),
        "p95_ms": round(sorted(timings_ms)[max(0, int(0.95 * len(timings_ms)) - 1)], 3),
        "min_ms": min(timings_ms),
        "max_ms": max(timings_ms),
        "samples_ms": timings_ms,
        "notes": (
            "GPU profile currently uses the same deterministic local pipeline path; "
            "replace with actual GPU-backed model execution in later optimization phase."
            if profile == "gpu"
            else "CPU baseline for current local inference stack."
        ),
    }


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Benchmark CPU/GPU runtime baselines for inference pipeline.")
    parser.add_argument("--runs", type=int, default=5, help="Number of timed runs per profile.")
    parser.add_argument("--duration-sec", type=int, default=60, help="Synthetic video duration per run.")
    parser.add_argument(
        "--output",
        type=str,
        default="outputs/benchmark_runtime.json",
        help="Benchmark JSON output path.",
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()

    cpu = _run_profile(profile="cpu", runs=max(1, args.runs), duration_sec=max(5, args.duration_sec))
    gpu = _run_profile(profile="gpu", runs=max(1, args.runs), duration_sec=max(5, args.duration_sec))

    benchmark_payload = {
        "benchmark": "phase7_inference_runtime",
        "profiles": [cpu, gpu],
    }

    output_path = Path(args.output)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(benchmark_payload, indent=2), encoding="utf-8")

    print(json.dumps(benchmark_payload, indent=2))


if __name__ == "__main__":
    main()



## File: backend/scripts/evaluate.py


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import sys


sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from training.trainer import evaluate_from_config


DEFAULT_CONFIG = Path("models/config/training_cpu.yaml")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Evaluate phase-5 multimodal scoring model.")
    parser.add_argument(
        "--config",
        type=str,
        default=str(DEFAULT_CONFIG),
        help="Path to training/evaluation config file.",
    )
    parser.add_argument(
        "--checkpoint",
        type=str,
        default="",
        help="Optional checkpoint path. If omitted, default checkpoint from config is used.",
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()
    checkpoint = args.checkpoint if args.checkpoint else None
    result = evaluate_from_config(config_path=args.config, checkpoint_path=checkpoint)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()



## File: backend/scripts/infer_cli.py


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import sys


sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from app.services.inference import InferenceService


VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".avi", ".webm"}


def _read_optional_text(path: str) -> str:
    if not path:
        return ""
    file_path = Path(path)
    if file_path.exists() and file_path.is_file():
        return file_path.read_text(encoding="utf-8", errors="ignore").strip()
    return ""


def _collect_batch_videos(batch_dir: str) -> list[Path]:
    root = Path(batch_dir)
    if not root.exists() or not root.is_dir():
        raise FileNotFoundError(f"Batch directory not found: {batch_dir}")
    videos = [p for p in root.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
    return sorted(videos)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Phase 6 CLI inference for startup pitch videos.")
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--video", type=str, default="", help="Single video path for inference.")
    mode.add_argument("--batch-dir", type=str, default="", help="Directory containing multiple videos.")

    parser.add_argument("--duration-sec", type=int, default=60, help="Video duration used for chunk timeline.")
    parser.add_argument("--language-hint", type=str, default="en-ta", help="Language hint passed to transcriber.")
    parser.add_argument("--title", type=str, default="", help="Optional title override for single-video mode.")
    parser.add_argument("--transcript-file", type=str, default="", help="Optional transcript .txt file for single mode.")
    parser.add_argument("--output", type=str, default="", help="Optional output JSON file path.")
    parser.add_argument(
        "--batch-output-dir",
        type=str,
        default="",
        help="Optional output directory for per-video JSON files in batch mode.",
    )
    return parser


def _run_single(args: argparse.Namespace, service: InferenceService) -> dict:
    transcript_text = _read_optional_text(args.transcript_file)
    title = args.title.strip() or Path(args.video).stem
    response = service.infer_video(
        video_path=args.video,
        duration_sec=args.duration_sec,
        transcript_text=transcript_text,
        title=title,
        language_hint=args.language_hint,
    )
    return response.model_dump()


def _run_batch(args: argparse.Namespace, service: InferenceService) -> dict:
    videos = _collect_batch_videos(args.batch_dir)
    batch_results: list[dict] = []

    output_dir = Path(args.batch_output_dir) if args.batch_output_dir else None
    if output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)

    for video_path in videos:
        response = service.infer_video(
            video_path=str(video_path),
            duration_sec=args.duration_sec,
            transcript_text="",
            title=video_path.stem,
            language_hint=args.language_hint,
        )
        payload = response.model_dump()
        batch_results.append(payload)

        if output_dir is not None:
            out_path = output_dir / f"{video_path.stem}.json"
            out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    return {
        "mode": "batch",
        "count": len(batch_results),
        "results": batch_results,
    }


def main() -> None:
    args = build_parser().parse_args()
    service = InferenceService()

    if args.video:
        output_payload = _run_single(args, service)
    else:
        output_payload = _run_batch(args, service)

    result_json = json.dumps(output_payload, indent=2)

    if args.output:
        output_path = Path(args.output)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(result_json, encoding="utf-8")

    print(result_json)


if __name__ == "__main__":
    main()



## File: backend/scripts/train.py


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
import sys


sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from training.trainer import train_from_config


DEFAULT_CONFIG = Path("models/config/training_cpu.yaml")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train phase-5 multimodal scoring model.")
    parser.add_argument(
        "--config",
        type=str,
        default=str(DEFAULT_CONFIG),
        help="Path to training config file.",
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()
    result = train_from_config(args.config)
    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()



## File: backend/tests/__init__.py


## File: backend/tests/test_api.py


In [ ]:
from fastapi.testclient import TestClient

from app.main import app
from app.schemas import PitchInput
from app.services.inference import InferenceService


def _sample_payload(title: str) -> dict:
    return {
        "title": title,
        "transcript": "",
        "language_hint": "en-ta",
        "presenter_profile": {"team": 3},
        "slide_text": ["Problem", "Solution", "Traction"],
        "video": {
            "file_name": f"{title.lower()}_pitch.mp4",
            "file_format": "mp4",
            "duration_sec": 90,
            "transcript_text": "We help local retailers with AI demand forecasting and bilingual advisory support.",
        },
        "slides": [
            {"title": "Problem", "content": "Inventory mismatch causes loss"},
            {"title": "Solution", "content": "Forecasting and automation"},
            {"title": "Traction", "content": "Pilot adoption in 12 stores"},
        ],
        "user_details": {
            "founder_name": "Founder",
            "startup_name": title,
            "sector": "RetailTech",
            "stage": "Seed",
        },
    }


def test_single_evaluation_endpoint() -> None:
    client = TestClient(app)
    response = client.post("/evaluate", json=_sample_payload("RetailPulse"))
    assert response.status_code == 200

    body = response.json()
    assert "request_id" in body
    assert 0 <= body["summary"]["overall_score"] <= 10
    assert body["summary"]["investment_band"] in {"high-potential", "watchlist", "early-risk"}
    assert body["summary"]["language_detected"] in {"en", "ta", "ta-en"}
    assert len(body["chunk_reports"]) > 0
    assert len(body["dashboard"]["quantitative_scores"]) == 10


def test_batch_evaluation_endpoint() -> None:
    client = TestClient(app)
    payload = {
        "pitches": [
            _sample_payload("RetailPulse"),
            _sample_payload("HealthMesh"),
        ]
    }

    response = client.post("/evaluate/batch", json=payload)
    assert response.status_code == 200

    body = response.json()
    assert len(body["evaluations"]) == 2


def test_batch_evaluation_requires_non_empty_pitches() -> None:
    client = TestClient(app)
    response = client.post("/evaluate/batch", json={"pitches": []})
    assert response.status_code == 400


def test_evaluate_endpoint_matches_shared_inference_output() -> None:
    payload_dict = _sample_payload("ParityCheck")

    client = TestClient(app)
    api_response = client.post("/evaluate", json=payload_dict)
    assert api_response.status_code == 200
    api_body = api_response.json()

    service = InferenceService()
    expected = service.evaluate_payload(PitchInput.model_validate(payload_dict)).model_dump()

    assert api_body == expected


## File: backend/tests/test_future_phases_contract.py


In [ ]:
from __future__ import annotations

from pathlib import Path

import pytest


BACKEND_ROOT = Path(__file__).resolve().parents[1]
APP_ROOT = BACKEND_ROOT / "app"
MODELS_ROOT = BACKEND_ROOT / "models"
TRAINING_ROOT = BACKEND_ROOT / "training"
SCRIPTS_ROOT = BACKEND_ROOT / "scripts"
TESTS_ROOT = BACKEND_ROOT / "tests"


def _require_paths_for_phase(phase_name: str, required: list[Path]) -> None:
    missing = [str(path.relative_to(BACKEND_ROOT)) for path in required if not path.exists()]
    if missing:
        pytest.skip(f"{phase_name} not implemented yet. Missing: {', '.join(missing)}")


def test_phase4_trainable_model_stack_contract() -> None:
    required = [
        MODELS_ROOT / "text_encoder.py",
        MODELS_ROOT / "visual_encoder.py",
        MODELS_ROOT / "audio_encoder.py",
        MODELS_ROOT / "fusion_head.py",
        MODELS_ROOT / "scoring_head.py",
        APP_ROOT / "services" / "extractors.py",
        APP_ROOT / "services" / "fusion.py",
        APP_ROOT / "services" / "scoring.py",
    ]
    _require_paths_for_phase("Phase 4", required)

    for path in required:
        assert path.is_file(), f"Expected file not found: {path}"

    model_files = [
        MODELS_ROOT / "text_encoder.py",
        MODELS_ROOT / "visual_encoder.py",
        MODELS_ROOT / "audio_encoder.py",
        MODELS_ROOT / "fusion_head.py",
        MODELS_ROOT / "scoring_head.py",
    ]
    for model_file in model_files:
        source = model_file.read_text(encoding="utf-8").lower()
        assert "class " in source, f"Expected at least one class in {model_file.name}"


def test_phase5_training_system_contract() -> None:
    required = [
        TRAINING_ROOT / "dataset_loader.py",
        TRAINING_ROOT / "trainer.py",
        TRAINING_ROOT / "metrics.py",
        TRAINING_ROOT / "losses.py",
        SCRIPTS_ROOT / "train.py",
        SCRIPTS_ROOT / "evaluate.py",
        MODELS_ROOT / "config" / "training_cpu.yaml",
        MODELS_ROOT / "config" / "training_gpu.yaml",
        MODELS_ROOT / "checkpoints",
    ]
    _require_paths_for_phase("Phase 5", required)

    for path in required:
        assert path.exists(), f"Expected training artifact missing: {path}"

    for script in [SCRIPTS_ROOT / "train.py", SCRIPTS_ROOT / "evaluate.py"]:
        source = script.read_text(encoding="utf-8").lower()
        assert "main" in source, f"Expected entrypoint-like function in {script.name}"


def test_phase6_cli_inference_contract() -> None:
    required = [
        SCRIPTS_ROOT / "infer_cli.py",
        APP_ROOT / "services" / "inference.py",
    ]
    _require_paths_for_phase("Phase 6", required)

    infer_cli_source = (SCRIPTS_ROOT / "infer_cli.py").read_text(encoding="utf-8").lower()
    assert "batch" in infer_cli_source, "Expected batch mode support in infer_cli.py"
    assert "json" in infer_cli_source, "Expected JSON output support in infer_cli.py"

    inference_source = (APP_ROOT / "services" / "inference.py").read_text(encoding="utf-8").lower()
    assert "transcrib" in inference_source
    assert "video" in inference_source


def test_phase6b_fastapi_cli_parity_contract() -> None:
    required = [
        APP_ROOT / "main.py",
        APP_ROOT / "services" / "inference.py",
    ]
    _require_paths_for_phase("Phase 6b", required)

    main_source = (APP_ROOT / "main.py").read_text(encoding="utf-8").lower()
    assert "/evaluate" in main_source
    assert "inference" in main_source, "Expected FastAPI to call shared inference service"


def test_phase7_quality_and_hardening_contract() -> None:
    benchmark_candidates = list(SCRIPTS_ROOT.glob("*benchmark*.py"))
    if not benchmark_candidates:
        pytest.skip("Phase 7 not implemented yet. Missing benchmark script in backend/scripts.")

    for benchmark_script in benchmark_candidates:
        assert benchmark_script.is_file()

    suite_text = "\n".join(path.read_text(encoding="utf-8", errors="ignore").lower() for path in TESTS_ROOT.glob("test_*.py"))
    assert "silent" in suite_text, "Expected silent audio edge-case tests"
    assert "short video" in suite_text or "short_video" in suite_text, "Expected short video edge-case tests"
    assert "corrupt" in suite_text, "Expected corrupt file edge-case tests"
    assert "tamil" in suite_text and "english" in suite_text, "Expected mixed English-Tamil tests"



## File: backend/tests/test_phase7_quality.py


In [ ]:
from __future__ import annotations

from app.pipeline import StartupPitchPipeline
from app.schemas import PitchInput, PitchVideoInput
from app.services.audio_processor import AudioChunkMetadata, AudioProcessor
from app.services.transcriber import build_local_transcriber
from app.services.video_processor import VideoProcessor


def test_silent_audio_chunk_uses_fallback_text() -> None:
    transcriber = build_local_transcriber(
        backend="faster-whisper",
        model_path="",
        min_audio_quality=0.35,
    )

    silent_metadata = AudioChunkMetadata(
        chunk_id=0,
        start_sec=0,
        end_sec=5,
        duration_sec=5.0,
        sample_rate=16000,
        num_samples=80000,
        mel_shape=(64, 156),
        mel_mean=0.0,
        mel_std=0.0,
        silence_ratio=0.99,
        clipping_ratio=0.0,
        audio_quality_score=0.85,
        audio_hash="silent-001",
        audio_file_path="audio/missing/chunk_0000.wav",
    )

    result = transcriber.transcribe_chunk(
        audio_file_path=silent_metadata.audio_file_path,
        audio_metadata=silent_metadata,
        fallback_text="",
        language_hint="en-ta",
    )

    assert result.status == "fallback-silent"
    assert result.text == "[inaudible chunk]"
    assert 0 <= result.confidence <= 1


def test_short_video_pipeline_still_returns_output() -> None:
    payload = PitchInput(
        title="Short Video",
        transcript="quick pitch for short video case",
        language_hint="en",
        video=PitchVideoInput(
            file_name="short_video.mp4",
            file_format="mp4",
            duration_sec=5,
            transcript_text="",
        ),
    )

    response = StartupPitchPipeline(window_seconds=5).evaluate(payload, request_id="short-video-case")

    assert response.request_id == "short-video-case"
    assert len(response.chunk_reports) == 1
    assert 0 <= response.summary.overall_score <= 10


def test_corrupt_video_name_is_handled_gracefully() -> None:
    payload = PitchInput(
        title="Corrupt File",
        transcript="This pitch should still produce a report even with corrupt file naming.",
        language_hint="en",
        video=PitchVideoInput(
            file_name="corrupt_file$$$.mp4",
            file_format="mp4",
            duration_sec=30,
            transcript_text="",
        ),
    )

    response = StartupPitchPipeline(window_seconds=5).evaluate(payload, request_id="corrupt-file-case")

    assert response.request_id == "corrupt-file-case"
    assert len(response.chunk_reports) > 0
    assert response.summary.investment_band in {"high-potential", "watchlist", "early-risk"}


def test_mixed_english_tamil_detection_returns_ta_en() -> None:
    mixed_english_tamil_text = "This startup helps farmers with crop pricing and தமிழ் சந்தை access."
    payload = PitchInput(
        title="English Tamil Mix",
        transcript=mixed_english_tamil_text,
        language_hint="en-ta",
        video=PitchVideoInput(
            file_name="eng_tamil_mix.mp4",
            file_format="mp4",
            duration_sec=10,
            transcript_text="",
        ),
    )

    response = StartupPitchPipeline(window_seconds=5).evaluate(payload, request_id="eng-ta-case")

    assert response.summary.language_detected == "ta-en"


def test_video_processor_and_audio_processor_metadata_shapes() -> None:
    video_processor = VideoProcessor(frame_extraction_enabled=False)
    audio_processor = AudioProcessor(audio_extraction_enabled=False)

    frame_meta = video_processor.extract_frames_for_chunk(
        video_file_path="demo.mp4",
        video_duration_sec=60,
        chunk_id=1,
        start_sec=5,
        end_sec=10,
    )
    audio_meta = audio_processor.extract_audio_chunk(
        video_file_path="demo.mp4",
        chunk_id=1,
        start_sec=5,
        end_sec=10,
    )

    assert frame_meta.frame_count == 5
    assert frame_meta.extraction_status == "skipped"
    assert audio_meta.sample_rate == 16000
    assert audio_meta.mel_shape[0] == 64
    assert 0 <= audio_meta.audio_quality_score <= 1



## File: backend/tests/test_pipeline.py


In [ ]:
from app.pipeline import StartupPitchPipeline
from app.schemas import PitchInput, PitchVideoInput, SlideInput, UserDetails


def test_pipeline_returns_scores_and_feedback() -> None:
    payload = PitchInput(
        title="AgriSage",
        transcript="",
        language_hint="en-ta",
        presenter_profile={"experience": "5 years"},
        slide_text=["Problem", "Solution", "Traction", "Business Model"],
        video=PitchVideoInput(
            file_name="agrisage_pitch.mp4",
            file_format="mp4",
            duration_sec=120,
            transcript_text="We solve crop loss for farmers using bilingual AI support. "
            "Our pilot improved yield and market access in Tamil Nadu.",
        ),
        slides=[
            SlideInput(title="Problem", content="Post-harvest crop loss is high"),
            SlideInput(title="Market", content="Large underserved smallholder segment"),
            SlideInput(title="Traction", content="Pilot growth and repeat usage"),
        ],
        user_details=UserDetails(
            founder_name="Aravind",
            startup_name="AgriSage",
            sector="AgriTech",
            stage="Seed",
        ),
    )

    response = StartupPitchPipeline(window_seconds=5).evaluate(payload, request_id="test-id")

    assert response.request_id == "test-id"
    assert 0 <= response.summary.overall_score <= 10
    assert 0 <= response.summary.confidence_score <= 10
    assert response.summary.language_detected in {"en", "ta", "ta-en"}
    assert len(response.chunk_reports) > 0
    assert len(response.dashboard.quantitative_scores) == 10



## File: backend/tests/test_transcriber.py


In [ ]:
from app.core.config import settings
from app.schemas import PitchInput, PitchVideoInput
from app.services.audio_processor import AudioChunkMetadata
from app.services.preprocessing import temporal_synchronize_and_segment
from app.services.transcriber import build_local_transcriber


def test_transcriber_falls_back_when_audio_file_missing() -> None:
    transcriber = build_local_transcriber(
        backend="faster-whisper",
        model_path="",
        min_audio_quality=0.35,
    )
    metadata = AudioChunkMetadata(
        chunk_id=0,
        start_sec=0,
        end_sec=5,
        duration_sec=5.0,
        sample_rate=16000,
        num_samples=80000,
        mel_shape=(64, 156),
        mel_mean=0.0,
        mel_std=0.0,
        silence_ratio=0.0,
        clipping_ratio=0.0,
        audio_quality_score=1.0,
        audio_hash="abc123",
        audio_file_path="audio/missing/chunk_0000.wav",
    )

    result = transcriber.transcribe_chunk(
        audio_file_path=metadata.audio_file_path,
        audio_metadata=metadata,
        fallback_text="deterministic fallback",
        language_hint="en-ta",
    )

    assert result.text == "deterministic fallback"
    assert result.status == "fallback-no-audio-file"
    assert 0 <= result.confidence <= 1


def test_preprocessing_uses_local_transcriber_with_fallback() -> None:
    previous_flag = settings.use_local_transcriber
    previous_backend = settings.local_transcriber_backend
    previous_model_path = settings.local_transcriber_model_path

    try:
        settings.use_local_transcriber = True
        settings.local_transcriber_backend = "faster-whisper"
        settings.local_transcriber_model_path = ""

        payload = PitchInput(
            title="FallbackTest",
            transcript="hello world this is stable fallback text",
            language_hint="en-ta",
            video=PitchVideoInput(
                file_name="fallback_test.mp4",
                duration_sec=10,
                transcript_text="",
            ),
        )

        chunks = temporal_synchronize_and_segment(payload, window_seconds=5)

        assert len(chunks) == 2
        assert all(chunk.text.strip() for chunk in chunks)
        assert chunks[0].alignment.text_excerpt == chunks[0].text
    finally:
        settings.use_local_transcriber = previous_flag
        settings.local_transcriber_backend = previous_backend
        settings.local_transcriber_model_path = previous_model_path



## File: backend/training/__init__.py


In [ ]:
"""Training package for phase 5 model development."""



## File: backend/training/dataset_loader.py


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
import random


FEATURE_DIM = 16
METRIC_COUNT = 10


@dataclass
class TrainingSample:
    features: list[float]
    metric_targets: list[float]
    overall_target: float


def _clamp_10(value: float) -> float:
    return max(0.0, min(10.0, value))


def _backend_root() -> Path:
    return Path(__file__).resolve().parents[1]


def _split_file(split: str) -> Path:
    return _backend_root() / "datasets" / "splits" / f"{split}.jsonl"


def _load_from_jsonl(path: Path) -> list[TrainingSample]:
    samples: list[TrainingSample] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        features = [float(v) for v in item.get("features", [])][:FEATURE_DIM]
        features = features + [0.0] * max(0, FEATURE_DIM - len(features))
        metrics = [float(v) for v in item.get("metric_targets", [])][:METRIC_COUNT]
        metrics = metrics + [0.0] * max(0, METRIC_COUNT - len(metrics))
        overall = float(item.get("overall_target", sum(metrics) / max(1, len(metrics))))
        samples.append(TrainingSample(features=features, metric_targets=metrics, overall_target=overall))
    return samples


def _synthetic_samples(count: int, seed: int) -> list[TrainingSample]:
    rng = random.Random(seed)
    hidden_w = [[rng.uniform(-0.8, 0.8) for _ in range(FEATURE_DIM)] for _ in range(METRIC_COUNT)]
    hidden_b = [rng.uniform(-0.7, 0.7) for _ in range(METRIC_COUNT)]

    samples: list[TrainingSample] = []
    for _ in range(count):
        features = [rng.uniform(0.0, 1.0) for _ in range(FEATURE_DIM)]
        metric_targets: list[float] = []
        for row, bias in zip(hidden_w, hidden_b):
            raw = sum(w * x for w, x in zip(row, features)) + bias + rng.uniform(-0.15, 0.15)
            scaled = 5.0 + raw * 2.2
            metric_targets.append(round(_clamp_10(scaled), 4))

        overall_target = round(sum(metric_targets) / len(metric_targets), 4)
        samples.append(
            TrainingSample(
                features=features,
                metric_targets=metric_targets,
                overall_target=overall_target,
            )
        )
    return samples


def load_split_samples(split: str, synthetic_count: int = 64, seed: int = 7) -> list[TrainingSample]:
    path = _split_file(split)
    if path.exists():
        loaded = _load_from_jsonl(path)
        if loaded:
            return loaded

    split_offset = {"train": 0, "val": 1000, "test": 2000}.get(split, 3000)
    return _synthetic_samples(count=synthetic_count, seed=seed + split_offset)



## File: backend/training/losses.py


In [ ]:
def mean_squared_error(predictions: list[float], targets: list[float]) -> float:
    if not predictions:
        return 0.0
    return sum((p - t) ** 2 for p, t in zip(predictions, targets)) / len(predictions)


def mse_gradient(prediction: float, target: float) -> float:
    return 2.0 * (prediction - target)



## File: backend/training/metrics.py


In [ ]:
import math


def mae(predictions: list[float], targets: list[float]) -> float:
    if not predictions:
        return 0.0
    return sum(abs(p - t) for p, t in zip(predictions, targets)) / len(predictions)


def rmse(predictions: list[float], targets: list[float]) -> float:
    if not predictions:
        return 0.0
    return math.sqrt(sum((p - t) ** 2 for p, t in zip(predictions, targets)) / len(predictions))


def spearman_rank_correlation(predictions: list[float], targets: list[float]) -> float:
    if len(predictions) < 2:
        return 0.0
    pred_sorted = sorted(range(len(predictions)), key=lambda i: predictions[i])
    target_sorted = sorted(range(len(targets)), key=lambda i: targets[i])

    pred_rank = {idx: rank for rank, idx in enumerate(pred_sorted)}
    target_rank = {idx: rank for rank, idx in enumerate(target_sorted)}

    n = len(predictions)
    diff_sq = sum((pred_rank[i] - target_rank[i]) ** 2 for i in range(n))
    return 1.0 - ((6.0 * diff_sq) / (n * (n**2 - 1)))



## File: backend/training/trainer.py


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
from typing import Callable

from training.dataset_loader import FEATURE_DIM, METRIC_COUNT, TrainingSample, load_split_samples
from training.losses import mean_squared_error, mse_gradient
from training.metrics import mae, rmse, spearman_rank_correlation


def _clamp_10(value: float) -> float:
    return max(0.0, min(10.0, value))


def _coerce_value(raw: str) -> bool | int | float | str:
    value = raw.strip()
    if value.lower() in {"true", "false"}:
        return value.lower() == "true"
    try:
        if "." in value:
            return float(value)
        return int(value)
    except ValueError:
        return value


def load_training_config(config_path: str) -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")

    parsed: dict = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        row = line.strip()
        if not row or row.startswith("#"):
            continue
        if ":" not in row:
            continue
        key, value = row.split(":", 1)
        parsed[key.strip()] = _coerce_value(value)

    defaults = {
        "epochs": 8,
        "learning_rate": 0.03,
        "train_samples": 120,
        "val_samples": 48,
        "test_samples": 48,
        "seed": 7,
        "checkpoint_dir": "models/checkpoints",
        "checkpoint_name": "phase5_checkpoint.json",
    }
    defaults.update(parsed)
    return defaults


@dataclass
class LinearMultiOutputModel:
    weights: list[list[float]]
    biases: list[float]

    @classmethod
    def initialize(cls, seed: int) -> "LinearMultiOutputModel":
        import random

        rng = random.Random(seed)
        weights = [[rng.uniform(-0.05, 0.05) for _ in range(FEATURE_DIM)] for _ in range(METRIC_COUNT)]
        biases = [0.0 for _ in range(METRIC_COUNT)]
        return cls(weights=weights, biases=biases)

    def predict_metrics(self, features: list[float]) -> list[float]:
        metrics: list[float] = []
        for row, bias in zip(self.weights, self.biases):
            raw = sum(w * x for w, x in zip(row, features)) + bias
            metrics.append(_clamp_10(5.0 + raw))
        return metrics

    def predict_overall(self, features: list[float]) -> float:
        metric_scores = self.predict_metrics(features)
        return sum(metric_scores) / len(metric_scores)


def _train_one_epoch(model: LinearMultiOutputModel, samples: list[TrainingSample], learning_rate: float) -> float:
    losses: list[float] = []
    for sample in samples:
        prediction = model.predict_metrics(sample.features)
        losses.append(mean_squared_error(prediction, sample.metric_targets))

        for out_idx in range(METRIC_COUNT):
            grad = mse_gradient(prediction[out_idx], sample.metric_targets[out_idx])
            for feat_idx in range(FEATURE_DIM):
                model.weights[out_idx][feat_idx] -= learning_rate * grad * sample.features[feat_idx]
            model.biases[out_idx] -= learning_rate * grad

    return sum(losses) / max(1, len(losses))


def _evaluate_regression(
    samples: list[TrainingSample],
    predictor: Callable[[list[float]], float],
) -> dict:
    preds = [predictor(sample.features) for sample in samples]
    targets = [sample.overall_target for sample in samples]
    return {
        "mae": round(mae(preds, targets), 4),
        "rmse": round(rmse(preds, targets), 4),
        "spearman": round(spearman_rank_correlation(preds, targets), 4),
    }


def _checkpoint_path(config: dict) -> Path:
    backend_root = Path(__file__).resolve().parents[1]
    checkpoint_dir = backend_root / str(config["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    return checkpoint_dir / str(config["checkpoint_name"])


def save_checkpoint(model: LinearMultiOutputModel, config: dict, history: list[dict]) -> Path:
    path = _checkpoint_path(config)
    payload = {
        "weights": model.weights,
        "biases": model.biases,
        "config": config,
        "history": history,
    }
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def load_checkpoint(checkpoint_path: str) -> LinearMultiOutputModel:
    payload = json.loads(Path(checkpoint_path).read_text(encoding="utf-8"))
    return LinearMultiOutputModel(weights=payload["weights"], biases=payload["biases"])


def train_from_config(config_path: str) -> dict:
    config = load_training_config(config_path)

    train_samples = load_split_samples(
        "train",
        synthetic_count=int(config["train_samples"]),
        seed=int(config["seed"]),
    )
    val_samples = load_split_samples(
        "val",
        synthetic_count=int(config["val_samples"]),
        seed=int(config["seed"]),
    )

    model = LinearMultiOutputModel.initialize(seed=int(config["seed"]))
    history: list[dict] = []

    for epoch in range(1, int(config["epochs"]) + 1):
        train_loss = _train_one_epoch(model, train_samples, float(config["learning_rate"]))
        val_metrics = _evaluate_regression(val_samples, model.predict_overall)
        history.append(
            {
                "epoch": epoch,
                "train_loss": round(train_loss, 4),
                "val_mae": val_metrics["mae"],
                "val_rmse": val_metrics["rmse"],
                "val_spearman": val_metrics["spearman"],
            }
        )

    checkpoint = save_checkpoint(model, config, history)

    return {
        "checkpoint_path": str(checkpoint),
        "epochs": int(config["epochs"]),
        "final_train_loss": history[-1]["train_loss"] if history else 0.0,
        "final_val_mae": history[-1]["val_mae"] if history else 0.0,
        "final_val_rmse": history[-1]["val_rmse"] if history else 0.0,
        "final_val_spearman": history[-1]["val_spearman"] if history else 0.0,
    }


def evaluate_from_config(config_path: str, checkpoint_path: str | None = None) -> dict:
    config = load_training_config(config_path)
    if checkpoint_path:
        model = load_checkpoint(checkpoint_path)
    else:
        default_path = _checkpoint_path(config)
        model = load_checkpoint(str(default_path))

    test_samples = load_split_samples(
        "test",
        synthetic_count=int(config["test_samples"]),
        seed=int(config["seed"]),
    )
    return _evaluate_regression(test_samples, model.predict_overall)



## File: convert_backend_to_notebooks.ps1


In [ ]:
$ErrorActionPreference = "Stop"

$root = Split-Path -Parent $MyInvocation.MyCommand.Path
$targetRoot = Join-Path $root "notebooks"
$outputPath = Join-Path $targetRoot "startup_pitch_evaluation_all.ipynb"

if (Test-Path $targetRoot) {
    Remove-Item -Path $targetRoot -Recurse -Force
}
New-Item -ItemType Directory -Path $targetRoot | Out-Null

function Get-LanguageForExtension {
    param ([string]$extension)

    $ext = $extension.ToLowerInvariant()
    switch ($ext) {
        ".py" { return "python" }
        ".ps1" { return "powershell" }
        ".json" { return "json" }
        ".yaml" { return "yaml" }
        ".yml" { return "yaml" }
        ".md" { return "markdown" }
        ".txt" { return "markdown" }
        ".toml" { return "toml" }
        ".ini" { return "ini" }
        ".cfg" { return "ini" }
        ".gitignore" { return "text" }
        ".gitattributes" { return "text" }
        ".csv" { return "csv" }
        ".tsv" { return "text" }
        ".log" { return "text" }
        default { return "text" }
    }
}

function Is-IncludedTextFile {
    param ([System.IO.FileInfo]$file)

    $name = $file.Name.ToLowerInvariant()
    $ext = $file.Extension.ToLowerInvariant()
    $allowByName = @("license", "readme", "plan.md", "missing.md")
    $allowByExtension = @(
        ".py", ".ps1", ".json", ".yaml", ".yml", ".md", ".txt", ".toml", ".ini", ".cfg", ".csv", ".tsv", ".log"
    )

    if ($allowByName -contains $name) {
        return $true
    }

    return $allowByExtension -contains $ext
}

function Get-NotebookSourceLines {
    param ([string]$content)

    $normalized = $content -replace "`r`n", "`n"
    if ([string]::IsNullOrEmpty($normalized)) {
        return @("`n")
    }

    $lines = $normalized -split "`n"
    $sourceLines = @()
    foreach ($line in $lines) {
        $sourceLines += ($line + "`n")
    }
    return $sourceLines
}

$excludedDirNames = @(".git", ".venv", "__pycache__", "notebooks", ".pytest_cache")

$candidateFiles = Get-ChildItem -Path $root -Recurse -File | Where-Object {
    $relative = $_.FullName.Substring($root.Length).TrimStart("\\")
    $segments = $relative.Split("\\")
    foreach ($segment in $segments) {
        if ($excludedDirNames -contains $segment.ToLowerInvariant()) {
            return $false
        }
    }
    return (Is-IncludedTextFile -file $_)
} | Sort-Object FullName

$cells = @()

$cells += [ordered]@{
    cell_type = "markdown"
    metadata = [ordered]@{ language = "markdown" }
    source = @(
        "# Startup Pitch Evaluation - Single Notebook Project`n",
        "This notebook contains a full project snapshot merged into one document.`n",
        "Each section starts with the source file path and then the file content.`n"
    )
}

foreach ($file in $candidateFiles) {
    $relativePath = $file.FullName.Substring($root.Length).TrimStart("\\") -replace "\\", "/"
    $extension = $file.Extension
    $language = Get-LanguageForExtension -extension $extension
    $content = Get-Content -Path $file.FullName -Raw -Encoding UTF8
    $sourceLines = Get-NotebookSourceLines -content $content

    $cells += [ordered]@{
        cell_type = "markdown"
        metadata = [ordered]@{ language = "markdown" }
        source = @(
            "## File: $relativePath`n"
        )
    }

    if ($language -eq "markdown") {
        $cells += [ordered]@{
            cell_type = "markdown"
            metadata = [ordered]@{ language = "markdown" }
            source = $sourceLines
        }
    }
    else {
        $cells += [ordered]@{
            cell_type = "code"
            execution_count = $null
            metadata = [ordered]@{ language = $language }
            outputs = @()
            source = $sourceLines
        }
    }
}

$notebook = [ordered]@{
    cells = $cells
    metadata = [ordered]@{
        kernelspec = [ordered]@{
            display_name = "Python 3"
            language = "python"
            name = "python3"
        }
        language_info = [ordered]@{
            name = "python"
            version = "3.11"
        }
    }
    nbformat = 4
    nbformat_minor = 5
}

$json = $notebook | ConvertTo-Json -Depth 100
Set-Content -Path $outputPath -Value $json -Encoding UTF8

Write-Output ("Converted {0} files into single notebook: {1}" -f $candidateFiles.Count, $outputPath)



## File: LICENSE


In [ ]:
MIT License

Copyright (c) 2026 Slncsmr

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.



## File: MISSING.md


Exactly the right question.  
Your current system is functionally complete by phases, but many parts are still lightweight placeholders versus the full neural architecture in your diagram.

**What Is Real And Working Now**

- End-to-end pipeline orchestration and output contracts are working in pipeline.py.
- API + CLI parity is working through shared inference in inference.py, main.py, and infer_cli.py.
- Local transcriber abstraction exists in transcriber.py, with fallback logic and backend switching.
- Phase 5/6/7 tests are passing, including edge cases and benchmark script in test_phase7_quality.py and benchmark_runtime.py.

**Replaced With Lightweight Versions (Not True Deep Models Yet)**

- Text, visual, audio “encoders” are deterministic heuristic/hash-style encoders, not neural encoders:
  - text_encoder.py
  - visual_encoder.py
  - audio_encoder.py
- Fusion is a weighted energy-based combiner, not cross-attention transformer fusion:
  - fusion_head.py
- Scoring head is rule-based linear mapping, not a learned deep hierarchical multi-head network:
  - scoring_head.py
- Training loop is a tiny linear multi-output learner with synthetic fallback data:
  - trainer.py
  - dataset_loader.py

**Still Missing For The Diagram-Level “Real Multimodal Neural System”**

- Real dataset and labels pipeline (Phase 1 was skipped), including actual train/val/test files and true labels.
- Real feature extraction from media:
  - Video processor currently metadata/stub only in video_processor.py
  - Audio processor currently metadata/stub only in audio_processor.py
- Real ASR execution from extracted chunk audio in normal flow:
  - Preprocessing calls transcriber, but chunk audio files are currently not truly extracted, so fallback path is common in preprocessing.py.
- Real deep training stack (PyTorch or similar):
  - No torch dependency in requirements.txt
  - No transformer/cross-attention training code, schedulers, mixed precision, or real GPU model graph.
- Real checkpoint format for neural nets:
  - Current checkpoints are JSON for linear weights, not deep model state_dict-style artifacts.

**So Is Neural Training Missing?**
Yes, true neural training is still missing.  
What you have now is a fast, testable scaffold that proves flow and contracts, not final model quality.

**Will Real Training Take Long?**
Yes, once you replace placeholders with true encoders + fusion network + real dataset:

- Small setup: tens of minutes to hours
- Medium realistic setup: hours to 1-2 days
- Larger tuning cycles: multiple days

If you want, next I can implement the first real neural step safely: add PyTorch-based model classes and trainer while preserving your current API/CLI contracts.



## File: PLAN.md


# Agent Work Plan

This file is the implementation backlog for coding agents.
Goal: build a fully local, self-trained multimodal video scoring system for startup pitch evaluation.

## Architecture Separation

CLI Path (Primary for ML)

- Training, evaluation, batch inference, data processing
- No HTTP dependency required
- Direct terminal commands in backend/scripts/

FastAPI Path (Optional for Frontend Only)

- Add only after Phase 6 is stable
- Thin wrapper around same core inference logic
- Both CLI and API return identical JSON outputs

## Global constraints

1. No external API calls for transcription, inference, or training.
2. Input must be video-first.
3. Transcription must run locally.
4. Final scores must come from self-trained models, not hardcoded heuristics.
5. CLI is primary for all ML development.
6. FastAPI is optional; add only for frontend integration at the end.
7. CLI and FastAPI share identical core inference code.

## Current baseline (already present)

1. Working backend scaffold and tests.
2. Heuristic extractors/fusion/scoring in service layer.
3. Structured response schema with chunk reports and dashboard payload.

## Phase-wise execution

## ~~Phase 0: Migration controls~~ ✓ COMPLETE

Tasks:

1. ~~Add feature flag `USE_HEURISTIC_PIPELINE` (default: true) for safe migration.~~
2. ~~Add feature flag `USE_LOCAL_TRANSCRIBER` (default: false until transcriber is ready).~~
3. ~~Add clear logging at pipeline start to show active mode.~~

Definition of done:

1. ~~Existing tests still pass.~~
2. ~~Pipeline can switch modes without runtime crash.~~

## Phase 1: Dataset and labels foundation

⏭️ SKIPPED - No dataset available yet. Proceeding with Phase 2 local preprocessing.

Tasks:

1. Create dataset folder structure:
   - `backend/datasets/raw`
   - `backend/datasets/processed`
   - `backend/datasets/splits`
   - `backend/datasets/labels`
2. Add label schema file `backend/datasets/labels/schema.json`.
3. Add split script `backend/scripts/make_splits.py` with speaker/startup leakage checks.
4. Add label validator `backend/scripts/validate_labels.py`.

Definition of done:

1. Split files generated for train/val/test.
2. Validation script passes on sample dataset.
3. No startup/speaker overlap across splits.

## ~~Phase 2: Local preprocessing from video~~ ✓ COMPLETE

Tasks:

1. ~~Add `backend/app/services/video_processor.py` for frame extraction per 5-second chunk.~~
2. ~~Add `backend/app/services/audio_processor.py` for audio extraction + mel features.~~
3. ~~Upgrade `backend/app/services/preprocessing.py` to use true timeline chunking from video duration.~~
4. ~~Persist chunk metadata for text/audio/visual alignment.~~

Definition of done:

1. ~~For any input video, aligned chunk artifacts are produced.~~
2. ~~Chunk metadata is deterministic.~~

## ~~Phase 3: Local transcription module~~ ✓ COMPLETE

Tasks:

1. ~~Add `backend/app/services/transcriber.py` abstraction interface.~~
2. ~~Implement local backend A: Whisper local.~~
3. ~~Implement local backend B: faster-whisper local.~~
4. ~~Add confidence and fallback behavior for empty/low-quality audio chunks.~~

Definition of done:

1. ~~No network usage during transcription.~~
2. ~~Stable transcript output for same input and settings.~~

## ~~Phase 4: Trainable multimodal model stack~~ ✓ COMPLETE

Tasks:

1. ~~Add model package:~~
   - `backend/models/text_encoder.py`
   - `backend/models/visual_encoder.py`
   - `backend/models/audio_encoder.py`
   - `backend/models/fusion_head.py`
   - `backend/models/scoring_head.py`
2. ~~Replace heuristics in `backend/app/services/extractors.py` with model inference wrappers.~~
3. ~~Replace fixed fusion in `backend/app/services/fusion.py` with learned fusion module.~~
4. ~~Replace weighted formula in `backend/app/services/scoring.py` with learned multi-output scoring.~~

Definition of done:

1. ~~Forward pass returns 10 metric scores, overall score, and confidence.~~
2. ~~Works on both CPU and GPU profiles.~~

## ~~Phase 5: Training system~~ ✓ COMPLETE

Tasks:

1. ~~Add training package:~~
   - `backend/training/dataset_loader.py`
   - `backend/training/trainer.py`
   - `backend/training/metrics.py`
   - `backend/training/losses.py`
2. ~~Add scripts:~~
   - `backend/scripts/train.py`
   - `backend/scripts/evaluate.py`
3. ~~Add config files:~~
   - `backend/models/config/training_cpu.yaml`
   - `backend/models/config/training_gpu.yaml`
4. ~~Add checkpoint saving/loading under `backend/models/checkpoints`.~~

Definition of done:

1. ~~Train run completes and writes checkpoints.~~
2. ~~Eval run outputs MAE/RMSE and ranking metrics.~~

## ~~Phase 6: CLI Inference Productization (ML development path)~~ ✓ COMPLETE

Tasks:

1. ~~Add `backend/scripts/infer_cli.py` for single video inference.~~
2. ~~Add batch mode for directory input.~~
3. ~~Extract core inference logic into `backend/app/services/inference.py`.~~
4. ~~Run full pipeline: video chunking -> local transcription -> multimodal model -> report JSON.~~

Definition of done:

1. ~~CLI produces full report JSON for unseen videos.~~
2. ~~CLI is the primary way ML engineers run inference and batch scoring.~~
3. ~~No FastAPI needed for ML development.~~

## ~~Phase 6b: FastAPI Frontend Integration (optional, after Phase 6 stable)~~ ✓ COMPLETE

Tasks:

1. ~~Keep `backend/app/main.py` as thin wrapper.~~
2. ~~POST /evaluate endpoint calls same `backend/app/services/inference.py` as CLI.~~
3. ~~Verify FastAPI output matches CLI output for identical inputs.~~
4. ~~This layer is optional for frontend UI integration only.~~

Definition of done:

1. ~~FastAPI endpoint returns identical JSON to CLI.~~
2. ~~Frontend can upload video and receive scoring report.~~
3. ~~FastAPI can be omitted entirely if not needed.~~

## ~~Phase 7: Quality and hardening~~ ✓ COMPLETE

Tasks:

1. ~~Expand tests:~~
   - unit tests for processors/transcriber/model wrappers
   - integration tests for end-to-end video -> score
2. ~~Add edge-case tests:~~
   - silent audio
   - short video
   - corrupt file
   - mixed English-Tamil speech
3. ~~Add benchmark script for CPU and GPU runtime.~~

Definition of done:

1. ~~Test suite passes.~~
2. ~~Failures are handled gracefully with clear error messages.~~
3. ~~Latency baseline is documented.~~

## Agent assignments (sequential)

1. Agent A: Dataset contracts, split tooling, label validator.
2. Agent B: Video/audio preprocessing modules and alignment metadata.
3. Agent C: Local transcription backends and abstraction.
4. Agent D: Model modules (text/visual/audio/fusion/scoring).
5. Agent E: Training loop, config, checkpoints, evaluation metrics.
6. Agent F: CLI inference script and shared core inference service (backend/app/services/inference.py).
7. Agent G: Testing, edge cases, benchmark, docs sync.
8. Agent H (optional): FastAPI wrapper for frontend (only after Phase 6 complete and stable).

## Completion criteria (Phase 0-7: ML development stable)

1. Input is video.
2. Transcription is local.
3. Scores are from self-trained models.
4. No external API calls in training/inference paths.
5. End-to-end CLI run: video -> final investor-ready report JSON.
6. All ML tasks (train, eval, inference) work from CLI without FastAPI.

## Frontend Integration (Phase 6b: optional, add after ML stable)

1. FastAPI wrapper added only after Phase 6 is tested and stable.
2. Single endpoint POST /evaluate receives same inputs as CLI.
3. FastAPI output is identical to CLI output for same inputs.
4. Frontend app can upload video and display scores/charts from FastAPI response.
5. FastAPI is removable without affecting ML pipeline.



## File: README.md


# Startup-Pitch-Evaluation

Multimodal AI backend starter project for evaluating startup pitch quality using text, visual, and audio feature pipelines.

This repository now includes a notebook-first project export under `notebooks/startup_pitch_evaluation_all.ipynb`, which merges the entire project into a single `.ipynb` file.

## What this project includes

- FastAPI service with evaluation endpoint
- Batch evaluation endpoint for multiple pitches
- Modular pipeline matching your architecture:
- Input preprocessing and temporal chunk synchronization (5s windows)
- Parallel feature extraction for text (NLP), visual (CV), and audio (DSP) signals
- Fusion and hierarchical scoring heads (10 quantitative metrics)
- Explainability outputs with modality attention weights per chunk
- Heuristic risk flag detection per chunk
- Automated strengths, weaknesses, suggestions, and dashboard-ready series output
- Unit test for end-to-end pipeline execution

## Architecture coverage (diagram to code)

- Input and preprocessing:
- Video, slides, and user details accepted through request schema
- Temporal synchronizer and segmenter in 5-second chunks
- Feature extraction (parallel):
- Text path: speech text input, EN/TA language detection, bilingual normalization, text embeddings
- Visual path: frame-level proxy extraction, delivery and confidence signals, visual embeddings
- Audio path: waveform/prosody proxy extraction, voice pace and prosody signals, audio embeddings
- Advanced fusion and scoring:
- Cross-modal weighted fusion (text/visual/audio attention weights)
- 10 scoring heads: 6 text-focused + 4 AV-focused metrics, each 0-10
- Output and reporting:
- Weighted aggregate score, investment band, risk flags, and automated feedback
- Investor dashboard payload for charts (metric series, modality weights, risk distribution)

## Project structure

```text
backend/
	app/
		core/config.py
		main.py
		pipeline.py
		schemas.py
		services/
			preprocessing.py
			extractors.py
			fusion.py
			scoring.py
			reporting.py
	tests/test_pipeline.py
	requirements.txt
notebooks/
	startup_pitch_evaluation_all.ipynb
```

## Quick start

1. Create and activate a Python virtual environment.
2. Install dependencies.
3. Run the API.

```powershell
cd backend
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
uvicorn app.main:app --reload
```

## Notebook-first quick start

Use this path if you want to operate the project as an `.ipynb` workflow.

```powershell
cd backend
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
python -m ipykernel install --user --name startup-pitch-eval --display-name "Python (startup-pitch-eval)"
cd ..
jupyter lab
```

Then open `notebooks/startup_pitch_evaluation_all.ipynb`.

- The notebook contains all project text/code files as sections.
- Each section starts with a file path heading and then the source content cell.

API will be available at:

- `http://127.0.0.1:8000/health`
- `http://127.0.0.1:8000/docs`

## Sample evaluation request

```powershell
curl -X POST "http://127.0.0.1:8000/evaluate" ^
	-H "Content-Type: application/json" ^
	-d "{\"title\":\"Startup Example A\",\"transcript\":\"\",\"language_hint\":\"en-ta\",\"presenter_profile\":{\"experience\":\"5 years\"},\"video\":{\"file_name\":\"startup_example_a_pitch.mp4\",\"file_format\":\"mp4\",\"duration_sec\":120,\"transcript_text\":\"We solve an industry workflow problem with bilingual AI support. Our pilot improved adoption and operational outcomes.\"},\"slides\":[{\"title\":\"Problem\",\"content\":\"Current workflow creates measurable loss\"},{\"title\":\"Market\",\"content\":\"Large underserved customer segment\"},{\"title\":\"Traction\",\"content\":\"Pilot growth and repeat usage\"}],\"user_details\":{\"founder_name\":\"Founder Example\",\"startup_name\":\"Startup Example A\",\"sector\":\"Industry Segment\",\"stage\":\"Seed\"}}"
```

## Sample batch request

```powershell
curl -X POST "http://127.0.0.1:8000/evaluate/batch" ^
	-H "Content-Type: application/json" ^
	-d "{\"pitches\":[{\"title\":\"Startup Example A\",\"transcript\":\"We solve a high-frequency operational problem for a target segment.\",\"language_hint\":\"en-ta\",\"slide_text\":[\"Problem\",\"Solution\"]},{\"title\":\"Startup Example B\",\"transcript\":\"We improve planning quality and reduce waste for local businesses.\",\"language_hint\":\"en-ta\",\"slide_text\":[\"Problem\",\"Traction\"]}]}"
```

## Output highlights

- `summary.investment_band`: `high-potential`, `watchlist`, or `early-risk`
- `summary.language_detected`: detected language profile (`en`, `ta`, `ta-en`)
- `chunk_reports[].attention`: text/visual/audio contribution weights
- `chunk_reports[].risk_flags`: detected risk hints such as overclaim or weak traction evidence
- `dashboard.quantitative_scores`: 10 chart-ready metrics for investor UI

## Contributing

Contributions are welcome for feature improvements, bug fixes, tests, documentation, and model integration.

### Development setup

1. Fork the repository.
2. Clone your fork and move to the backend directory.
3. Create and activate a virtual environment.
4. Install dependencies and run tests.

```powershell
git clone <your-fork-url>
cd Startup-Pitch-Evaluation/backend
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
pytest -q
```

### Contribution workflow

1. Create a feature branch from main.
2. Make focused changes with clear commit messages.
3. Add or update tests for all behavior changes.
4. Ensure all tests pass locally.
5. Open a pull request with a concise summary.

### Pull request checklist

- Code is readable and scoped to one logical change.
- New or changed behavior is covered by tests.
- README or docs are updated when API behavior changes.
- No sensitive data, secrets, or private files are committed.

### Coding guidelines

- Keep API schemas backward-compatible when possible.
- Prefer small, composable service functions.
- Use clear naming for scoring and reporting outputs.
- Keep sample data generic and free of personal identifiers.

## Notes

- Current feature extraction and fusion are deterministic placeholder implementations designed for fast iteration.
- You can replace individual service modules with real model inference (Whisper, wav2vec2, ViT, multimodal transformers) without changing API contracts.

## Run tests

```powershell
cd backend
pytest -q
```

